## Versi 1

In [8]:
import re
import string
import emoji
import pandas as pd

# --- Fungsi bantu ---
def extract_links(text):
    return re.findall(r"https?://\S+", text)

def preprocess_text(text):
    text = str(text).lower()

    # Simpan link sementara
    links = extract_links(text)

    # Ganti link dengan token sementara
    for i, link in enumerate(links):
        text = text.replace(link, f"LINK{i}")

    # Hapus timestamp seperti 11:35 atau 07:06
    text = re.sub(r"\b\d{1,2}:\d{2}\b", "", text)

    # Hapus tanda baca, kecuali @ dan ?
    punctuation = string.punctuation.replace("@", "").replace("?", "")
    text = re.sub(rf"[{re.escape(punctuation)}]", " ", text)

    # Bersihkan spasi
    text = re.sub(r"\s+", " ", text).strip()
    return text

def is_only_emoji(text):
    return all(char in emoji.EMOJI_DATA for char in text.strip()) and len(text.strip()) > 0

def is_only_numbers(text):
    return re.fullmatch(r"\d+", text.strip()) is not None

def is_only_mentions(text):
    return bool(re.fullmatch(r"(@[^\s@]+[\s]*)+", text.strip()))

def is_single_word(text):
    return len(text.strip().split()) == 1

def is_only_mentions_and_emoji(text):
    words = text.strip().split()
    if not words:
        return False
    for word in words:
        if not (word.startswith("@") or all(char in emoji.EMOJI_DATA for char in word)):
            return False
    return True

def is_only_mentions_links_and_emoji(text):
    words = text.strip().split()
    if not words:
        return False
    for word in words:
        if word.startswith("@"):
            continue
        elif re.match(r"https?://\S+", word):  # link asli
            continue
        elif word.lower().startswith("link"):  # token LINK0, LINK1, dll
            continue
        elif all(char in emoji.EMOJI_DATA for char in word):
            continue
        else:
            return False
    return True

def get_removal_reason(text):
    if is_only_mentions_links_and_emoji(text):
        return "mention_dan_link_atau_emoji_saja"
    elif is_only_mentions_and_emoji(text):
        return "mention_dan_emoji_saja"
    elif is_only_emoji(text):
        return "emoji_saja"
    elif is_only_numbers(text):
        return "angka_saja"
    elif is_only_mentions(text):
        return "mention_saja"
    elif is_single_word(text):
        return "satu_kata_saja"
    else:
        return None

# --- Contoh komentar untuk diuji ---
sample_comments = [
    "https://chat.whatsapp.com/EvrzHUhRpOIG6Uimgnli1W masuk sini grup prediksi parlayyyy" # kombinasi emoji + mention + link
]

# --- Uji proses ---
results = []

for comment in sample_comments:
    cleaned = preprocess_text(comment)
    reason = get_removal_reason(cleaned)
    status = "✅ dipertahankan" if reason is None else f"🗑️ dibuang ({reason})"

    results.append({
        "Komentar Asli": comment,
        "Komentar Dibersihkan": cleaned,
        "Status": status
    })

# Tampilkan hasil
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))


                                                                      Komentar Asli                     Komentar Dibersihkan          Status
https://chat.whatsapp.com/EvrzHUhRpOIG6Uimgnli1W masuk sini grup prediksi parlayyyy LINK0 masuk sini grup prediksi parlayyyy ✅ dipertahankan


## Tes versi pakai

In [118]:
# pakai
import re
import string
import emoji
import unicodedata
import pandas as pd


# --- Fungsi bantu ---
def extract_links(text):
    return re.findall(r"https?://\S+", text)

# Mapping karakter non-Latin (Cyrillic) yang mirip huruf Latin
visual_map = {
    'А': 'A', 'В': 'B', 'С': 'C', 'Е': 'E', 'Н': 'H',
    'К': 'K', 'М': 'M', 'О': 'O', 'Р': 'P', 'Т': 'T',
    'Х': 'X', 'а': 'a', 'е': 'e', 'о': 'o', 'р': 'p',
    'с': 'c', 'у': 'y', 'х': 'x',
}

def merge_split_letters(text):
    """
    Gabungkan kembali huruf-huruf satuan yang dipisah spasi.
    Contoh: 'ᴡ ᴀ ᴋ ᴀ ᴛ ᴏ ᴛ ᴏ' → 'ᴡᴀᴋᴀᴛᴏᴛᴏ'
    """
    tokens = text.strip().split()
    merged = []
    temp = ""

    for token in tokens:
        if len(token) == 1 and unicodedata.category(token).startswith("L"):
            temp += token
        else:
            if temp:
                merged.append(temp)
                temp = ""
            merged.append(token)

    if temp:
        merged.append(temp)

    return " ".join(merged)


def preprocess_text(text):
    """
    Normalisasi teks komentar:
    - huruf kecil
    - link → LINK0, LINK1, ...
    - hapus timestamp (hh:mm)
    - hapus tanda baca ASCII (kecuali @ dan ?)
    - hapus karakter non-latin dan non-emoji dari tiap kata
    - rapikan spasi
    """
    text = str(text).lower()

    # Ganti semua link dengan token LINK0, LINK1, ...
    links = extract_links(text)
    for i, link in enumerate(links):
        text = text.replace(link, f"LINK{i}")

    # Hapus format jam (misalnya 11:35)
    text = re.sub(r"\b\d{1,2}:\d{2}\b", "", text)

    # Hapus tanda baca ASCII (kecuali @ dan ?)
    ascii_punctuation = ''.join(c for c in string.punctuation if c not in "@?")
    text = re.sub(rf"[{re.escape(ascii_punctuation)}]", " ", text)

    # Ganti karakter non-printable
    text = re.sub(r"[\u0000-\u001F\u007F-\u009F\u200B-\u200D]+", " ", text)

    def is_valid_text_char(char):
        # PERBAIKAN: Memastikan return False berada di dalam blok try-except
        try:
            name = unicodedata.name(char)
            # Hanya huruf/angka Latin atau gaya Latin matematika (bold/italic/fancy)
            if "LATIN" in name or "MATHEMATICAL" in name or "DIGIT" in name:
                return True
        except ValueError:
            # Jika karakter tidak punya nama (misalnya, beberapa simbol), anggap tidak valid
            return False
        # Jika punya nama tapi tidak cocok kriteria di atas, anggap tidak valid
        return False

    def clean_word(word):
        if word.lower().startswith("link"):
            return word

        chars_to_remove = {'♥', '❤', '💖', '💘', '💗', '💓', '💝'}  # simbol love
        result = ""

        if word.startswith("@"):
            # Pecah dengan regex: @username + sisa karakter
            # Pecah @username dari sisanya
            match = re.match(r"^@[\w\d_]+", word) # match @username
            if match:
                username = match.group()
                tail = word[len(username):]
            else:
                username = "@"
                tail = word[1:]
            
            # Bersihkan tail: buang simbol non-emoji (kecuali emoji tetap)
            # Sisakan hanya emoji dari tail
            tail_cleaned = ''.join(
                c for c in tail if c not in chars_to_remove and c in emoji.EMOJI_DATA
            )

            return f"{username} {tail_cleaned}".strip()

        for char in word:
            if char in chars_to_remove:
                continue
            if char in visual_map:
                result += visual_map[char]
            elif is_valid_text_char(char) or char in "@?" or char in emoji.EMOJI_DATA:
                result += char

        return result





    words = text.strip().split()
    words = [clean_word(word) for word in words if clean_word(word)]
    text = " ".join(words)
    text = merge_split_letters(text)  # ⬅️ Tambahkan ini untuk gabungkan huruf visual
    
    # PERBAIKAN: Memastikan fungsi selalu mengembalikan string
    return text



def is_only_emoji(text):
    return all(char in emoji.EMOJI_DATA for char in text.strip()) and len(text.strip()) > 0

def is_only_numbers(text):
    return re.fullmatch(r"\d+", text.strip()) is not None

def is_only_mentions(text):
    return bool(re.fullmatch(r"(@[^\s@]+[\s]*)+", text.strip()))

def is_single_word(text):
    return len(text.strip().split()) == 1

def is_only_mentions_and_emoji(text):
    words = text.strip().split()
    if not words:
        return False
    for word in words:
        if word.startswith("@"):
            continue
        elif all(char in emoji.EMOJI_DATA for char in word):
            continue
        else:
            return False
    return True

def is_only_mentions_links_and_emoji(text):
    words = text.strip().split()
    if not words:
        return False
    for word in words:
        if word.startswith("@"):
            continue
        elif re.match(r"https?://\S+", word):
            continue
        elif word.lower().startswith("link"):
            continue
        elif all(char in emoji.EMOJI_DATA for char in word):
            continue
        else:
            return False
    return True

def is_only_symbols_or_emoticons(text):
    """
    Buang komentar jika:
    - Tidak mengandung huruf Latin (a-z)
    - Terdiri dari karakter simbol, bentuk dekoratif, atau emoticon teks
    """
    text = text.strip()
    if not text:
        return False

    # Jika ada huruf Latin, anggap ada kata bermakna
    if re.search(r"[a-z]", text):
        return False

    # Jika panjang teks minimal dan semua bukan angka/alfabet → buang
    if len(text) == 0: return False # Hindari ZeroDivisionError
    non_alnum_ratio = sum(1 for c in text if not c.isalnum()) / len(text)
    return non_alnum_ratio > 0.6  # 60% lebih bukan huruf/angka

def is_mention_and_short_word(text):
    """
    Buang komentar jika hanya terdiri dari mention + satu kata pendek (panjang ≤ 3 huruf).
    Contoh: "@3 am", "@akun wkwk"
    """
    tokens = text.strip().split()
    if len(tokens) == 2 and tokens[0].startswith("@"):
        word = tokens[1]
        return len(word) <= 5
    return False

# def is_dominantly_mention(text):
#     """
#     Jika 70% atau lebih token adalah mention (@...), buang.
#     Kecuali jika ada kalimat panjang setelahnya.
#     """
    
#     tokens = text.strip().split()
#     if not tokens:
#         return False
#     mention_count = sum(1 for token in tokens if token.startswith("@"))
#     ratio = mention_count / len(tokens)
#     return ratio >= 0.66 and len(tokens) <= 5
def is_dominantly_mention(text):
    """
    Buang komentar jika:
    - ≥ 2 mention
    - dan sisa token tidak mengandung kata panjang bermakna
    - dan total token ≤ 6
    """
    tokens = text.strip().split()
    if not tokens:
        return False

    mention_tokens = [t for t in tokens if t.startswith("@")]
    non_mentions = [t for t in tokens if not t.startswith("@")]

    if len(mention_tokens) < 2 or len(tokens) > 6:
        return False

    # Jika semua non-mention pendek (≤4) dan bukan emoji
    semua_pendek = all(
        len(t) <= 4 and not all(char in emoji.EMOJI_DATA for char in t)
        for t in non_mentions
    )

    return semua_pendek






def get_removal_reason(text):
    # PERBAIKAN: Menambahkan pengecekan tipe data untuk keamanan
    if not isinstance(text, str) or not text.strip():
        return "kosong_setelah_pembersihan"
    elif is_only_mentions_links_and_emoji(text):
        return "mention_dan_link_atau_emoji_saja"
    elif is_only_mentions_and_emoji(text):
        return "mention_dan_emoji_saja"
    elif is_only_emoji(text):
        return "emoji_saja"
    elif is_only_symbols_or_emoticons(text):
        return "simbol_non_teks"
    elif is_only_numbers(text):
        return "angka_saja"
    elif is_only_mentions(text):
        return "mention_saja"
    elif is_mention_and_short_word(text):
        return "mention_dan_kata_pendek"
    elif is_single_word(text):
        return "satu_kata_saja"
    elif is_dominantly_mention(text):
        return "mayoritas_mention"
    elif is_mention_and_short_word(text):
        return "mention_dan_kata_pendek"

    else:
        return None



# --- Komentar untuk diuji langsung ---
sample_comments = [
    "₍ ˃ᯅ˂ （ ͜•人 ͜•） • ‿ώ‿ ꪊꪻ",
    "mampir dong",
    "fyp gara² ͜•人 ͜•）",
    "₍ ˃ᯅ˂ （ ͜•人 ͜•） • ‿ώ‿ ꪊꪻ",
    "@˚✧ u t a r i ✧˚ @liga @cel",
    "@✨ @tino☄️🪐 ⚔️",
    "fyp gara² ͜•人 ͜•）",
    "@𝙈𝙖𝙥𝙡𝙖𝙮𝟕𝟖𝟗 💚congratulations💚",
    "pasti fyp karena ini ͜•人 ͜•）",
    "@macanhoki789꧂💛💜never give up brother💜💛",
    "gemes ͜•人 ͜•）",
    "𝗟𝗘𝗦𝗧𝗜𝟳𝟳💥 gacir, pertama kali awak jepe 10 juta langsung jozz! 🚀",
    "Baru coba ✌𝐌𝐀𝐍𝘿𝘼𝙇𝙄𝙆𝘼 𝟕𝟕✌, ternyata prosesnya beneran cepet!",
    "𝐆𝐒𝐎 𝟕𝟖𝟗 🥰💚 slalu baik bosku 💚🥰",
    "Sumpah penasaran, kok pada bahas 🐊M̲A̲N̲D̲A̲L̲I̲K̲A̲7̲7̲🐊 di kolom komentar?",
    "07:10 ✌ ABCD4D ✌ selalu memanjakan semua pemain !!",
    "Kaget sih awalnya aku kira bohongan,ternyata di ♥M♥A♥N♥D♥A♥L♥I♥K♥A♥7♥7♥ banyak banget benefitnya!.",
    "Udah paling top markotop cuman di █▓▒▒░░░MANDALIKA77░░░▒▒▓█.",
    "Yang butuh hiburan di ᆞMᆞᆞAᆞᆞNᆞᆞDᆞᆞAᆞᆞLᆞᆞIᆞᆞKᆞᆞAᆞᆞ7ᆞᆞ7ᆞ",
    "Cepet ajak temen kalian di M̳̿͟͞A̳̿͟͞N̳̿͟͞D̳̿͟͞A̳̿͟͞L̳̿͟͞I̳̿͟͞K̳̿͟͞A̳̿͟͞7̳̿͟͞7̳̿͟͞ bisa dapet saldo cuy",
    "Di 𝐷𝑂𝑅𝘈𝟳7 saya menemukan kesempatan, dari penjaja es kelapa jadi pemilik usaha makanan.",
    "Salut bang keren kontennya , salam Jepe DI ⭐𝗪𝗘𝗧𝗢𝗡𝟴𝟴",
    "🔥𝗕𝗔𝗡𝗝𝗔𝗥𝟰𝗗",
    "Update mbak4d2 terbaru hadir! deposit qris 1 detik langsung masuk !! keren sekarang!",
    "ini sih king aloy sama si 𝗕𝗥𝗢𝗪𝗜𝗡𝟰𝗗 sama sama kocak ya...mantap banget sihh di 𝗕𝗥𝗢𝗪𝗜𝗡𝟰𝗗 𝗕𝗢𝗡𝗨𝗦 𝟭𝟬𝟬% 𝗨𝗡𝗧𝗨𝗞 𝗡𝗘𝗪 𝗠𝗘𝗠𝗕𝗘𝗥 𝗕𝗘𝗕𝗔𝗦 𝗜𝗣 semoga makin jaya ya",
    "20:33 buset dah  ─═ 𝗕 𝗘 𝗥 𝗞 𝗔 𝗛 𝟵 𝟵 ═─",
    "YANG INI KAK PASTI WD , 𝘭𝘪𝘯𝘬𝘯𝘺𝘢 𝘣𝘪𝘴𝘢 𝘬𝘦𝘵𝘪𝘬 𝘥𝘪 𝘨𝘰𝘰𝘨𝘭𝘦 : 🔍 𝗧𝗛𝗢𝗥𝟯𝟭𝟭",
    "Iseng-iseng main А𝙀𝘙𝘖𝟪𝟾, malah jadi hobi, cuan terus!",
    "ᴡ ᴀ ᴋ ᴀ ᴛ ᴏ ᴛ ᴏ BAGI THR LEEE",
    "𝚁𝙼𝙰789 🌠⭐sehat selalu ketua🌠⭐7",
    "axl777 bangbang888 🥰",
    "AHMA𝘋𝙏O𝐓O aku cba, eh lngsg dapet cuan gede",
    "@pgc789🖤✈️ so happy king✈️✈️🏁🏁",
    "☯️🆔jp good ❤️pgc789 ❤️ happy always boskuh🇦🇱🇦🇱🇦🇱506",
    "🔥🐲🔥 anubis303 ❤️‍🔥jp mexwin❤️‍🔥 bahagia selalu ketua 🎊",
    "@ch4lox 11🪐 @m4k4n d4l4m🐉 @rkdkroken",
    "@struggle£ •",
    "@3 am",
    "@desawa @adeknya liam @change exe™",
    "@netral23 @tuan muda",
    "@★★★ @aditya saputra @okan1 @pemburu janda isi saldo dulu",
    "@bh 12 LINK0",
    "🥰✨🙏thankssss 🌸🀄️uburwiner🀄️🌸 bahagia selalu bossku 💮🉐035826",
    "@boyzxzy @라잘리✨ @ kenzchan ✓ @tokisaki kurumi🗿 @diyaa🤍 @slim shady2695 ehem ehem oto",
    "@s ᴋ ᴇ ᴘ ᴛ ɪ s @apisss @ibeng",
    "@@st• ar°ga",
    "@violettaメ♪♪ @? @may๑˙❥˙๑ @danzz hyper @s t i k m a n🦹 @afra @sara",
    "@tana ♥︎",
    "@lu² @ql6 @zynthiaè 🦋 @★ daayy 𐙚 @chaaa @giegie",
    "@giiee •ムイチロー @yoichi ಠ∀ಠ @ ☆naa aja☆ @mᥲᥣᥣᥕᥫ᭡ 🗿🗿",
]

# --- Proses dan tampilkan hasil ---
results = []
for comment in sample_comments:
    cleaned = preprocess_text(comment)
    reason = get_removal_reason(cleaned)
    status = "✅ Dipertahankan" if reason is None else f"🗑️ Dibuang ({reason})"
    results.append({
        "Asli": comment,
        "Bersih": cleaned,
        "Status": status
    })

# Tampilkan hasil dalam bentuk tabel
df = pd.DataFrame(results)
print(df.to_string(index=False))


                                                                                                                                            Asli                                                                                                                                        Bersih                                        Status
                                                                                                                        ₍ ˃ᯅ˂ （ ͜•人 ͜•） • ‿ώ‿ ꪊꪻ                                                                                                                                                     🗑️ Dibuang (kosong_setelah_pembersihan)
                                                                                                                                     mampir dong                                                                                                                                   mampir dong                               ✅ Dipertahankan
 

In [ ]:
# versi coba coba(tidak pakai) 
import re
import string
import emoji
import unicodedata
import pandas as pd

# Mapping karakter visual Cyrillic → Latin
visual_map = {
    'А': 'A', 'В': 'B', 'С': 'C', 'Е': 'E', 'Н': 'H',
    'К': 'K', 'М': 'M', 'О': 'O', 'Р': 'P', 'Т': 'T',
    'Х': 'X', 'а': 'a', 'е': 'e', 'о': 'o', 'р': 'p',
    'с': 'c', 'у': 'y', 'х': 'x',
}

# Simbol love yang harus dihapus
chars_to_remove = {'♥', '❤', '💖', '💘', '💗', '💓', '💝'}

def extract_links(text):
    return re.findall(r"https?://\S+", text)

def extract_all_emojis(text):
    """Ambil semua emoji kompleks (termasuk ❤️, 🇮🇩)"""
    return [e["emoji"] for e in emoji.emoji_list(text)]

def is_valid_text_char(char):
    try:
        name = unicodedata.name(char)
        return any(keyword in name for keyword in ["LATIN", "MATHEMATICAL", "DIGIT"])
    except ValueError:
        return False

def merge_split_letters(text):
    """Gabungkan huruf-huruf satuan yang dipisah spasi"""
    tokens = text.strip().split()
    merged = []
    temp = ""
    for token in tokens:
        if len(token) == 1 and unicodedata.category(token).startswith("L"):
            temp += token
        else:
            if temp:
                merged.append(temp)
                temp = ""
            merged.append(token)
    if temp:
        merged.append(temp)
    return " ".join(merged)

def clean_word(word, emoji_found):
    if word.lower().startswith("link"):
        return word

    if word.startswith("@"):
        match = re.match(r"^@[\w\d_]+", word)
        if match:
            username = match.group()
            tail = word[len(username):]
        else:
            username = "@"
            tail = word[1:]
        tail_cleaned = ''.join(
            c for c in tail if c not in chars_to_remove and c in emoji_found
        )
        return f"{username} {tail_cleaned}".strip()

    result = ""
    for char in word:
        if char in chars_to_remove:
            continue
        if char in visual_map:
            result += visual_map[char]
        elif is_valid_text_char(char) or char in "@?" or char in emoji_found:
            result += char
    return result

def preprocess_text(text):
    text = str(text)
    emoji_found = extract_all_emojis(text)
    text = text.lower()

    # Tokenisasi link
    links = extract_links(text)
    for i, link in enumerate(links):
        text = text.replace(link, f"LINK{i}")

    # Hapus timestamp seperti 11:35
    text = re.sub(r"\b\d{1,2}:\d{2}\b", "", text)

    # Hapus tanda baca ASCII kecuali @ dan ?
    ascii_punctuation = ''.join(c for c in string.punctuation if c not in "@?")
    text = re.sub(rf"[{re.escape(ascii_punctuation)}]", " ", text)

    # Karakter kontrol dan invisible
    text = re.sub(r"[\u0000-\u001F\u007F-\u009F\u200B-\u200D]+", " ", text)

    # Bersihkan per kata
    words = text.strip().split()
    words = [clean_word(word, emoji_found) for word in words if clean_word(word, emoji_found)]
    text = " ".join(words)

    # Gabungkan huruf-huruf satuan
    text = merge_split_letters(text)
    return text


# --- Fungsi klasifikasi buang/tidak ---
def is_only_emoji(text):
    return all(char in emoji.EMOJI_DATA for char in text.strip()) and len(text.strip()) > 0

def is_only_numbers(text):
    return re.fullmatch(r"\d+", text.strip()) is not None

def is_only_mentions(text):
    return bool(re.fullmatch(r"(@[^\s@]+[\s]*)+", text.strip()))

def is_single_word(text):
    return len(text.strip().split()) == 1

def is_only_mentions_and_emoji(text):
    words = text.strip().split()
    return all(word.startswith("@") or all(char in emoji.EMOJI_DATA for char in word) for word in words)

def is_only_mentions_links_and_emoji(text):
    words = text.strip().split()
    return all(
        word.startswith("@")
        or word.lower().startswith("link")
        or re.match(r"https?://\S+", word)
        or all(char in emoji.EMOJI_DATA for char in word)
        for word in words
    )

def is_only_symbols_or_emoticons(text):
    if not text.strip():
        return False
    if re.search(r"[a-z]", text):
        return False
    non_alnum_ratio = sum(1 for c in text if not c.isalnum()) / len(text)
    return non_alnum_ratio > 0.6

def get_removal_reason(text):
    if not text.strip():
        return "kosong_setelah_pembersihan"
    elif is_only_mentions_links_and_emoji(text):
        return "mention_dan_link_atau_emoji_saja"
    elif is_only_mentions_and_emoji(text):
        return "mention_dan_emoji_saja"
    elif is_only_emoji(text):
        return "emoji_saja"
    elif is_only_symbols_or_emoticons(text):
        return "simbol_non_teks"
    elif is_only_numbers(text):
        return "angka_saja"
    elif is_only_mentions(text):
        return "mention_saja"
    elif is_single_word(text):
        return "satu_kata_saja"
    else:
        return None

# --- Contoh Pengujian ---
if __name__ == "__main__":
    sample_comments = [
    "₍ ˃ᯅ˂ （ ͜•人 ͜•） • ‿ώ‿ ꪊꪻ",
    "mampir dong",
    "fyp gara² ͜•人 ͜•）",
    "₍ ˃ᯅ˂ （ ͜•人 ͜•） • ‿ώ‿ ꪊꪻ",
    "@˚✧ u t a r i ✧˚ @liga @cel",
    "@✨ @tino☄️🪐 ⚔️",
    "fyp gara² ͜•人 ͜•）",
    "@𝙈𝙖𝙥𝙡𝙖𝙮𝟕𝟖𝟗 💚congratulations💚",
    "pasti fyp karena ini ͜•人 ͜•）",
    "@macanhoki789꧂💛💜never give up brother💜💛",
    "gemes ͜•人 ͜•）",
    "𝗟𝗘𝗦𝗧𝗜𝟳𝟳💥 gacir, pertama kali awak jepe 10 juta langsung jozz! 🚀",
    "Baru coba ✌𝐌𝐀𝐍𝘿𝘼𝙇𝙄𝙆𝘼 𝟕𝟕✌, ternyata prosesnya beneran cepet!",
    "𝐆𝐒𝐎 𝟕𝟖𝟗 🥰💚 slalu baik bosku 💚🥰",
    "Sumpah penasaran, kok pada bahas 🐊M̲A̲N̲D̲A̲L̲I̲K̲A̲7̲7̲🐊 di kolom komentar?",
    "07:10 ✌ ABCD4D ✌ selalu memanjakan semua pemain !!",
    "Kaget sih awalnya aku kira bohongan,ternyata di ♥M♥A♥N♥D♥A♥L♥I♥K♥A♥7♥7♥ banyak banget benefitnya!.",
    "Udah paling top markotop cuman di █▓▒▒░░░MANDALIKA77░░░▒▒▓█.",
    "Yang butuh hiburan di ᆞMᆞᆞAᆞᆞNᆞᆞDᆞᆞAᆞᆞLᆞᆞIᆞᆞKᆞᆞAᆞᆞ7ᆞᆞ7ᆞ",
    "Cepet ajak temen kalian di M̳̿͟͞A̳̿͟͞N̳̿͟͞D̳̿͟͞A̳̿͟͞L̳̿͟͞I̳̿͟͞K̳̿͟͞A̳̿͟͞7̳̿͟͞7̳̿͟͞ bisa dapet saldo cuy",
    "Di 𝐷𝑂𝑅𝘈𝟳7 saya menemukan kesempatan, dari penjaja es kelapa jadi pemilik usaha makanan.",
    "Salut bang keren kontennya , salam Jepe DI ⭐𝗪𝗘𝗧𝗢𝗡𝟴𝟴",
    "🔥𝗕𝗔𝗡𝗝𝗔𝗥𝟰𝗗",
    "Update mbak4d2 terbaru hadir! deposit qris 1 detik langsung masuk !! keren sekarang!",
    "ini sih king aloy sama si 𝗕𝗥𝗢𝗪𝗜𝗡𝟰𝗗 sama sama kocak ya...mantap banget sihh di 𝗕𝗥𝗢𝗪𝗜𝗡𝟰𝗗 𝗕𝗢𝗡𝗨𝗦 𝟭𝟬𝟬% 𝗨𝗡𝗧𝗨𝗞 𝗡𝗘𝗪 𝗠𝗘𝗠𝗕𝗘𝗥 𝗕𝗘𝗕𝗔𝗦 𝗜𝗣 semoga makin jaya ya",
    "20:33 buset dah  ─═ 𝗕 𝗘 𝗥 𝗞 𝗔 𝗛 𝟵 𝟵 ═─",
    "YANG INI KAK PASTI WD , 𝘭𝘪𝘯𝘬𝘯𝘺𝘢 𝘣𝘪𝘴𝘢 𝘬𝘦𝘵𝘪𝘬 𝘥𝘪 𝘨𝘰𝘰𝘨𝘭𝘦 : 🔍 𝗧𝗛𝗢𝗥𝟯𝟭𝟭",
    "Iseng-iseng main А𝙀𝘙𝘖𝟪𝟾, malah jadi hobi, cuan terus!",
    "ᴡ ᴀ ᴋ ᴀ ᴛ ᴏ ᴛ ᴏ BAGI THR LEEE",
    "𝚁𝙼𝙰789 🌠⭐sehat selalu ketua🌠⭐7",
    "axl777 bangbang888 🥰",
    "AHMA𝘋𝙏O𝐓O aku cba, eh lngsg dapet cuan gede",
    "@pgc789🖤✈️ so happy king✈️✈️🏁🏁",
    "☯️🆔jp good ❤️pgc789 ❤️ happy always boskuh🇦🇱🇦🇱🇦🇱506",
]

    results = []
    for comment in sample_comments:
        cleaned = preprocess_text(comment)
        reason = get_removal_reason(cleaned)
        status = "✅ Dipertahankan" if reason is None else f"🗑️ Dibuang ({reason})"
        results.append({
            "Asli": comment,
            "Bersih": cleaned,
            "Status": status
        })

    df = pd.DataFrame(results)
    print(df.to_string(index=False))


                                                                                                                                            Asli                                                                                                                                        Bersih                                        Status
                                                                                                                        ₍ ˃ᯅ˂ （ ͜•人 ͜•） • ‿ώ‿ ꪊꪻ                                                                                                                                                     🗑️ Dibuang (kosong_setelah_pembersihan)
                                                                                                                                     mampir dong                                                                                                                                   mampir dong                               ✅ Dipertahankan
 

## Versi yang di pakai

In [45]:
import pandas as pd
import re
import string
import emoji

# --- Fungsi bantu ---

def extract_links(text):
    # Mengambil semua link dalam teks
    return re.findall(r"https?://\S+", text)

def preprocess_text(text):
    """
    Melakukan normalisasi teks komentar:
    - huruf kecil
    - link diganti token LINK0, LINK1, dst
    - hapus timestamp seperti 11:35
    - hapus tanda baca (kecuali @ dan ?)
    - rapikan spasi
    """
    text = str(text).lower()
    
    # Simpan link asli untuk ditandai dengan token
    links = extract_links(text)

    # Ganti semua link dengan token LINK0, LINK1, ...
    for i, link in enumerate(links):
        text = text.replace(link, f"LINK{i}")

    # Hapus format jam (misalnya 11:35)
    text = re.sub(r"\b\d{1,2}:\d{2}\b", "", text)

    # Hapus semua tanda baca kecuali @ dan ?
    punctuation = string.punctuation.replace("@", "").replace("?", "")
    text = re.sub(rf"[{re.escape(punctuation)}]", " ", text)

    # Rapikan spasi yang berlebihan
    text = re.sub(r"\s+", " ", text).strip()
    return text

def is_only_emoji(text):
    # Deteksi jika seluruh teks hanya emoji
    return all(char in emoji.EMOJI_DATA for char in text.strip()) and len(text.strip()) > 0

def is_only_numbers(text):
    # Deteksi jika teks hanya angka
    return re.fullmatch(r"\d+", text.strip()) is not None

def is_only_mentions(text):
    # Deteksi jika teks hanya berisi mention seperti @user @admin
    return bool(re.fullmatch(r"(@[^\s@]+[\s]*)+", text.strip()))

def is_single_word(text):
    # Deteksi jika hanya terdiri dari 1 kata
    return len(text.strip().split()) == 1

def is_only_mentions_and_emoji(text):
    """
    Deteksi komentar yang hanya berisi kombinasi:
    - mention
    - emoji
    Tidak ada kata bermakna lain
    """
    words = text.strip().split()
    if not words:
        return False

    for word in words:
        if word.startswith("@"):
            continue
        elif all(char in emoji.EMOJI_DATA for char in word):
            continue
        else:
            return False
    return True

def is_only_mentions_links_and_emoji(text):
    """
    Deteksi komentar yang hanya berisi kombinasi dari:
    - mention
    - emoji
    - link (baik masih asli maupun sudah jadi LINK0, LINK1, ...)
    """
    words = text.strip().split()
    if not words:
        return False

    for word in words:
        if word.startswith("@"):
            continue
        elif re.match(r"https?://\S+", word):  # link asli (seandainya belum ditoken)
            continue
        elif word.lower().startswith("link"):  # token link (LINK0, LINK1, ...)
            continue
        elif all(char in emoji.EMOJI_DATA for char in word):
            continue
        else:
            return False
    return True

def is_only_symbols_or_emoticons(text):
    """
    Buang komentar jika:
    - Tidak mengandung huruf Latin (a-z)
    - Terdiri dari karakter simbol, bentuk dekoratif, atau emoticon teks
    """
    text = text.strip()
    if not text:
        return False

    # Jika ada huruf Latin, anggap ada kata bermakna
    if re.search(r"[a-z]", text):
        return False

    # Jika panjang teks minimal dan semua bukan angka/alfabet → buang
    non_alnum_ratio = sum(1 for c in text if not c.isalnum()) / len(text)
    return non_alnum_ratio > 0.6  # 60% lebih bukan huruf/angka

def get_removal_reason(text):
    """
    Memberikan alasan kenapa komentar dibuang
    Jika tidak memenuhi kriteria pembuangan → return None
    """
    if is_only_mentions_links_and_emoji(text):
        return "mention_dan_link_atau_emoji_saja"
    elif is_only_mentions_and_emoji(text):
        return "mention_dan_emoji_saja"
    elif is_only_emoji(text):
        return "emoji_saja"
    elif is_only_symbols_or_emoticons(text):
        return "simbol_non_teks"
    elif is_only_numbers(text):
        return "angka_saja"
    elif is_only_mentions(text):
        return "mention_saja"
    elif is_single_word(text):
        return "satu_kata_saja"
    else:
        return None



# --- Muat dan proses data ---

# Ganti path file sesuai lokasi datamu
INPUT_FILE = "D:/kuliah/Skripsi/clean-code/final_results(1).csv"
OUTPUT_CLEAN = "cleaned_comments_filtered(fr1c).csv"
OUTPUT_REMOVED = "log_removed_comments(fr1c).csv"

# Baca file CSV dan buang baris dengan komentar kosong
df = pd.read_csv(INPUT_FILE)
df = df.dropna(subset=["T_komentar"])

# Lakukan preprocessing terhadap kolom komentar
df["T_komentar"] = df["T_komentar"].apply(preprocess_text)

# Tandai komentar yang perlu dibuang beserta alasannya
df["removal_reason"] = df["T_komentar"].apply(get_removal_reason)

# Bagi data menjadi dua: bersih dan dibuang
df_cleaned = df[df["removal_reason"].isna()].copy()      # yang tidak dibuang
df_removed = df[df["removal_reason"].notna()].copy()     # yang dibuang

# Simpan hasil ke file CSV
df_cleaned.drop(columns=["removal_reason"]).to_csv(OUTPUT_CLEAN, index=False)
df_removed.to_csv(OUTPUT_REMOVED, index=False)

# Ringkasan hasil
print(f"✅ Total komentar bersih: {len(df_cleaned)} (disimpan di {OUTPUT_CLEAN})")
print(f"🗑️ Total komentar dibuang: {len(df_removed)} (disimpan di {OUTPUT_REMOVED})")


✅ Total komentar bersih: 20593 (disimpan di cleaned_comments_filtered(fr1c).csv)
🗑️ Total komentar dibuang: 9597 (disimpan di log_removed_comments(fr1c).csv)


# versi fix

In [119]:
# pakai
import re
import string
import emoji
import unicodedata
import pandas as pd


# --- Fungsi bantu ---
def extract_links(text):
    return re.findall(r"https?://\S+", text)

# Mapping karakter non-Latin (Cyrillic) yang mirip huruf Latin
visual_map = {
    'А': 'A', 'В': 'B', 'С': 'C', 'Е': 'E', 'Н': 'H',
    'К': 'K', 'М': 'M', 'О': 'O', 'Р': 'P', 'Т': 'T',
    'Х': 'X', 'а': 'a', 'е': 'e', 'о': 'o', 'р': 'p',
    'с': 'c', 'у': 'y', 'х': 'x',
}

def merge_split_letters(text):
    """
    Gabungkan kembali huruf-huruf satuan yang dipisah spasi.
    Contoh: 'ᴡ ᴀ ᴋ ᴀ ᴛ ᴏ ᴛ ᴏ' → 'ᴡᴀᴋᴀᴛᴏᴛᴏ'
    """
    tokens = text.strip().split()
    merged = []
    temp = ""

    for token in tokens:
        if len(token) == 1 and unicodedata.category(token).startswith("L"):
            temp += token
        else:
            if temp:
                merged.append(temp)
                temp = ""
            merged.append(token)

    if temp:
        merged.append(temp)

    return " ".join(merged)


def preprocess_text(text):
    """
    Normalisasi teks komentar:
    - huruf kecil
    - link → LINK0, LINK1, ...
    - hapus timestamp (hh:mm)
    - hapus tanda baca ASCII (kecuali @ dan ?)
    - hapus karakter non-latin dan non-emoji dari tiap kata
    - rapikan spasi
    """
    text = str(text).lower()

    # Ganti semua link dengan token LINK0, LINK1, ...
    links = extract_links(text)
    for i, link in enumerate(links):
        text = text.replace(link, f"LINK{i}")

    # Hapus format jam (misalnya 11:35)
    text = re.sub(r"\b\d{1,2}:\d{2}\b", "", text)

    # Hapus tanda baca ASCII (kecuali @ dan ?)
    ascii_punctuation = ''.join(c for c in string.punctuation if c not in "@?")
    text = re.sub(rf"[{re.escape(ascii_punctuation)}]", " ", text)

    # Ganti karakter non-printable
    text = re.sub(r"[\u0000-\u001F\u007F-\u009F\u200B-\u200D]+", " ", text)

    def is_valid_text_char(char):
        # PERBAIKAN: Memastikan return False berada di dalam blok try-except
        try:
            name = unicodedata.name(char)
            # Hanya huruf/angka Latin atau gaya Latin matematika (bold/italic/fancy)
            if "LATIN" in name or "MATHEMATICAL" in name or "DIGIT" in name:
                return True
        except ValueError:
            # Jika karakter tidak punya nama (misalnya, beberapa simbol), anggap tidak valid
            return False
        # Jika punya nama tapi tidak cocok kriteria di atas, anggap tidak valid
        return False

    def clean_word(word):
        if word.lower().startswith("link"):
            return word

        chars_to_remove = {'♥', '❤', '💖', '💘', '💗', '💓', '💝'}  # simbol love
        result = ""

        if word.startswith("@"):
            # Pecah dengan regex: @username + sisa karakter
            # Pecah @username dari sisanya
            match = re.match(r"^@[\w\d_]+", word) # match @username
            if match:
                username = match.group()
                tail = word[len(username):]
            else:
                username = "@"
                tail = word[1:]
            
            # Bersihkan tail: buang simbol non-emoji (kecuali emoji tetap)
            # Sisakan hanya emoji dari tail
            tail_cleaned = ''.join(
                c for c in tail if c not in chars_to_remove and c in emoji.EMOJI_DATA
            )

            return f"{username} {tail_cleaned}".strip()

        for char in word:
            if char in chars_to_remove:
                continue
            if char in visual_map:
                result += visual_map[char]
            elif is_valid_text_char(char) or char in "@?" or char in emoji.EMOJI_DATA:
                result += char

        return result





    words = text.strip().split()
    words = [clean_word(word) for word in words if clean_word(word)]
    text = " ".join(words)
    text = merge_split_letters(text)  # ⬅️ Tambahkan ini untuk gabungkan huruf visual
    
    # PERBAIKAN: Memastikan fungsi selalu mengembalikan string
    return text



def is_only_emoji(text):
    return all(char in emoji.EMOJI_DATA for char in text.strip()) and len(text.strip()) > 0

def is_only_numbers(text):
    return re.fullmatch(r"\d+", text.strip()) is not None

def is_only_mentions(text):
    return bool(re.fullmatch(r"(@[^\s@]+[\s]*)+", text.strip()))

def is_single_word(text):
    return len(text.strip().split()) == 1

def is_only_mentions_and_emoji(text):
    words = text.strip().split()
    if not words:
        return False
    for word in words:
        if word.startswith("@"):
            continue
        elif all(char in emoji.EMOJI_DATA for char in word):
            continue
        else:
            return False
    return True

def is_only_mentions_links_and_emoji(text):
    words = text.strip().split()
    if not words:
        return False
    for word in words:
        if word.startswith("@"):
            continue
        elif re.match(r"https?://\S+", word):
            continue
        elif word.lower().startswith("link"):
            continue
        elif all(char in emoji.EMOJI_DATA for char in word):
            continue
        else:
            return False
    return True

def is_only_symbols_or_emoticons(text):
    """
    Buang komentar jika:
    - Tidak mengandung huruf Latin (a-z)
    - Terdiri dari karakter simbol, bentuk dekoratif, atau emoticon teks
    """
    text = text.strip()
    if not text:
        return False

    # Jika ada huruf Latin, anggap ada kata bermakna
    if re.search(r"[a-z]", text):
        return False

    # Jika panjang teks minimal dan semua bukan angka/alfabet → buang
    if len(text) == 0: return False # Hindari ZeroDivisionError
    non_alnum_ratio = sum(1 for c in text if not c.isalnum()) / len(text)
    return non_alnum_ratio > 0.6  # 60% lebih bukan huruf/angka

def is_mention_and_short_word(text):
    """
    Buang komentar jika hanya terdiri dari mention + satu kata pendek (panjang ≤ 3 huruf).
    Contoh: "@3 am", "@akun wkwk"
    """
    tokens = text.strip().split()
    if len(tokens) == 2 and tokens[0].startswith("@"):
        word = tokens[1]
        return len(word) <= 5
    return False

# def is_dominantly_mention(text):
#     """
#     Jika 70% atau lebih token adalah mention (@...), buang.
#     Kecuali jika ada kalimat panjang setelahnya.
#     """
    
#     tokens = text.strip().split()
#     if not tokens:
#         return False
#     mention_count = sum(1 for token in tokens if token.startswith("@"))
#     ratio = mention_count / len(tokens)
#     return ratio >= 0.66 and len(tokens) <= 5
def is_dominantly_mention(text):
    """
    Buang komentar jika:
    - ≥ 2 mention
    - dan sisa token tidak mengandung kata panjang bermakna
    - dan total token ≤ 6
    """
    tokens = text.strip().split()
    if not tokens:
        return False

    mention_tokens = [t for t in tokens if t.startswith("@")]
    non_mentions = [t for t in tokens if not t.startswith("@")]

    if len(mention_tokens) < 2 or len(tokens) > 6:
        return False

    # Jika semua non-mention pendek (≤4) dan bukan emoji
    semua_pendek = all(
        len(t) <= 4 and not all(char in emoji.EMOJI_DATA for char in t)
        for t in non_mentions
    )

    return semua_pendek






def get_removal_reason(text):
    # PERBAIKAN: Menambahkan pengecekan tipe data untuk keamanan
    if not isinstance(text, str) or not text.strip():
        return "kosong_setelah_pembersihan"
    elif is_only_mentions_links_and_emoji(text):
        return "mention_dan_link_atau_emoji_saja"
    elif is_only_mentions_and_emoji(text):
        return "mention_dan_emoji_saja"
    elif is_only_emoji(text):
        return "emoji_saja"
    elif is_only_symbols_or_emoticons(text):
        return "simbol_non_teks"
    elif is_only_numbers(text):
        return "angka_saja"
    elif is_only_mentions(text):
        return "mention_saja"
    elif is_mention_and_short_word(text):
        return "mention_dan_kata_pendek"
    elif is_single_word(text):
        return "satu_kata_saja"
    elif is_dominantly_mention(text):
        return "mayoritas_mention"
    elif is_mention_and_short_word(text):
        return "mention_dan_kata_pendek"

    else:
        return None
    
    
# --- Muat dan proses data ---

# Ganti path file sesuai lokasi datamu
INPUT_FILE = "D:/kuliah/Skripsi/clean-code/final_results(1).csv"
OUTPUT_CLEAN = "cleaned_comments_filtered(fr1d).csv"
OUTPUT_REMOVED = "log_removed_comments(fr1d).csv"

# Baca file CSV dan buang baris dengan komentar kosong
df = pd.read_csv(INPUT_FILE)
df = df.dropna(subset=["T_komentar"])

# Lakukan preprocessing terhadap kolom komentar
df["T_komentar"] = df["T_komentar"].apply(preprocess_text)

# Tandai komentar yang perlu dibuang beserta alasannya
df["removal_reason"] = df["T_komentar"].apply(get_removal_reason)

# Bagi data menjadi dua: bersih dan dibuang
df_cleaned = df[df["removal_reason"].isna()].copy()      # yang tidak dibuang
df_removed = df[df["removal_reason"].notna()].copy()     # yang dibuang

# Simpan hasil ke file CSV
df_cleaned.drop(columns=["removal_reason"]).to_csv(OUTPUT_CLEAN, index=False)
df_removed.to_csv(OUTPUT_REMOVED, index=False)

# Ringkasan hasil
print(f"✅ Total komentar bersih: {len(df_cleaned)} (disimpan di {OUTPUT_CLEAN})")
print(f"🗑️ Total komentar dibuang: {len(df_removed)} (disimpan di {OUTPUT_REMOVED})")


✅ Total komentar bersih: 20349 (disimpan di cleaned_comments_filtered(fr1d).csv)
🗑️ Total komentar dibuang: 9841 (disimpan di log_removed_comments(fr1d).csv)


In [128]:
# final_results(2).csv

# pakai
import re
import string
import emoji
import unicodedata
import pandas as pd


# --- Fungsi bantu ---
def extract_links(text):
    return re.findall(r"https?://\S+", text)

# Mapping karakter non-Latin (Cyrillic) yang mirip huruf Latin
visual_map = {
    'А': 'A', 'В': 'B', 'С': 'C', 'Е': 'E', 'Н': 'H',
    'К': 'K', 'М': 'M', 'О': 'O', 'Р': 'P', 'Т': 'T',
    'Х': 'X', 'а': 'a', 'е': 'e', 'о': 'o', 'р': 'p',
    'с': 'c', 'у': 'y', 'х': 'x',
}

def merge_split_letters(text):
    """
    Gabungkan kembali huruf-huruf satuan yang dipisah spasi.
    Contoh: 'ᴡ ᴀ ᴋ ᴀ ᴛ ᴏ ᴛ ᴏ' → 'ᴡᴀᴋᴀᴛᴏᴛᴏ'
    """
    tokens = text.strip().split()
    merged = []
    temp = ""

    for token in tokens:
        if len(token) == 1 and unicodedata.category(token).startswith("L"):
            temp += token
        else:
            if temp:
                merged.append(temp)
                temp = ""
            merged.append(token)

    if temp:
        merged.append(temp)

    return " ".join(merged)


def preprocess_text(text):
    """
    Normalisasi teks komentar:
    - huruf kecil
    - link → LINK0, LINK1, ...
    - hapus timestamp (hh:mm)
    - hapus tanda baca ASCII (kecuali @ dan ?)
    - hapus karakter non-latin dan non-emoji dari tiap kata
    - rapikan spasi
    """
    text = str(text).lower()

    # Ganti semua link dengan token LINK0, LINK1, ...
    links = extract_links(text)
    for i, link in enumerate(links):
        text = text.replace(link, f"LINK{i}")

    # Hapus format jam (misalnya 11:35)
    text = re.sub(r"\b\d{1,2}:\d{2}\b", "", text)

    # Hapus tanda baca ASCII (kecuali @ dan ?)
    ascii_punctuation = ''.join(c for c in string.punctuation if c not in "@?")
    text = re.sub(rf"[{re.escape(ascii_punctuation)}]", " ", text)

    # Ganti karakter non-printable
    text = re.sub(r"[\u0000-\u001F\u007F-\u009F\u200B-\u200D]+", " ", text)

    def is_valid_text_char(char):
        # PERBAIKAN: Memastikan return False berada di dalam blok try-except
        try:
            name = unicodedata.name(char)
            # Hanya huruf/angka Latin atau gaya Latin matematika (bold/italic/fancy)
            if "LATIN" in name or "MATHEMATICAL" in name or "DIGIT" in name:
                return True
        except ValueError:
            # Jika karakter tidak punya nama (misalnya, beberapa simbol), anggap tidak valid
            return False
        # Jika punya nama tapi tidak cocok kriteria di atas, anggap tidak valid
        return False

    def clean_word(word):
        if word.lower().startswith("link"):
            return word

        chars_to_remove = {'♥', '❤', '💖', '💘', '💗', '💓', '💝'}  # simbol love
        result = ""

        if word.startswith("@"):
            # Pecah dengan regex: @username + sisa karakter
            # Pecah @username dari sisanya
            match = re.match(r"^@[\w\d_]+", word) # match @username
            if match:
                username = match.group()
                tail = word[len(username):]
            else:
                username = "@"
                tail = word[1:]
            
            # Bersihkan tail: buang simbol non-emoji (kecuali emoji tetap)
            # Sisakan hanya emoji dari tail
            tail_cleaned = ''.join(
                c for c in tail if c not in chars_to_remove and c in emoji.EMOJI_DATA
            )

            return f"{username} {tail_cleaned}".strip()

        for char in word:
            if char in chars_to_remove:
                continue
            if char in visual_map:
                result += visual_map[char]
            elif is_valid_text_char(char) or char in "@?" or char in emoji.EMOJI_DATA:
                result += char

        return result





    words = text.strip().split()
    words = [clean_word(word) for word in words if clean_word(word)]
    text = " ".join(words)
    text = merge_split_letters(text)  # ⬅️ Tambahkan ini untuk gabungkan huruf visual
    
    # PERBAIKAN: Memastikan fungsi selalu mengembalikan string
    return text



def is_only_emoji(text):
    return all(char in emoji.EMOJI_DATA for char in text.strip()) and len(text.strip()) > 0

def is_only_numbers(text):
    return re.fullmatch(r"\d+", text.strip()) is not None

def is_only_mentions(text):
    return bool(re.fullmatch(r"(@[^\s@]+[\s]*)+", text.strip()))

def is_single_word(text):
    return len(text.strip().split()) == 1

def is_only_mentions_and_emoji(text):
    words = text.strip().split()
    if not words:
        return False
    for word in words:
        if word.startswith("@"):
            continue
        elif all(char in emoji.EMOJI_DATA for char in word):
            continue
        else:
            return False
    return True

def is_only_mentions_links_and_emoji(text):
    words = text.strip().split()
    if not words:
        return False
    for word in words:
        if word.startswith("@"):
            continue
        elif re.match(r"https?://\S+", word):
            continue
        elif word.lower().startswith("link"):
            continue
        elif all(char in emoji.EMOJI_DATA for char in word):
            continue
        else:
            return False
    return True

def is_only_symbols_or_emoticons(text):
    """
    Buang komentar jika:
    - Tidak mengandung huruf Latin (a-z)
    - Terdiri dari karakter simbol, bentuk dekoratif, atau emoticon teks
    """
    text = text.strip()
    if not text:
        return False

    # Jika ada huruf Latin, anggap ada kata bermakna
    if re.search(r"[a-z]", text):
        return False

    # Jika panjang teks minimal dan semua bukan angka/alfabet → buang
    if len(text) == 0: return False # Hindari ZeroDivisionError
    non_alnum_ratio = sum(1 for c in text if not c.isalnum()) / len(text)
    return non_alnum_ratio > 0.6  # 60% lebih bukan huruf/angka

def is_mention_and_short_word(text):
    """
    Buang komentar jika hanya terdiri dari mention + satu kata pendek (panjang ≤ 3 huruf).
    Contoh: "@3 am", "@akun wkwk"
    """
    tokens = text.strip().split()
    if len(tokens) == 2 and tokens[0].startswith("@"):
        word = tokens[1]
        return len(word) <= 5
    return False

# def is_dominantly_mention(text):
#     """
#     Jika 70% atau lebih token adalah mention (@...), buang.
#     Kecuali jika ada kalimat panjang setelahnya.
#     """
    
#     tokens = text.strip().split()
#     if not tokens:
#         return False
#     mention_count = sum(1 for token in tokens if token.startswith("@"))
#     ratio = mention_count / len(tokens)
#     return ratio >= 0.66 and len(tokens) <= 5
def is_dominantly_mention(text):
    """
    Buang komentar jika:
    - ≥ 2 mention
    - dan sisa token tidak mengandung kata panjang bermakna
    - dan total token ≤ 6
    """
    tokens = text.strip().split()
    if not tokens:
        return False

    mention_tokens = [t for t in tokens if t.startswith("@")]
    non_mentions = [t for t in tokens if not t.startswith("@")]

    if len(mention_tokens) < 2 or len(tokens) > 6:
        return False

    # Jika semua non-mention pendek (≤4) dan bukan emoji
    semua_pendek = all(
        len(t) <= 4 and not all(char in emoji.EMOJI_DATA for char in t)
        for t in non_mentions
    )

    return semua_pendek






def get_removal_reason(text):
    # PERBAIKAN: Menambahkan pengecekan tipe data untuk keamanan
    if not isinstance(text, str) or not text.strip():
        return "kosong_setelah_pembersihan"
    elif is_only_mentions_links_and_emoji(text):
        return "mention_dan_link_atau_emoji_saja"
    elif is_only_mentions_and_emoji(text):
        return "mention_dan_emoji_saja"
    elif is_only_emoji(text):
        return "emoji_saja"
    elif is_only_symbols_or_emoticons(text):
        return "simbol_non_teks"
    elif is_only_numbers(text):
        return "angka_saja"
    elif is_only_mentions(text):
        return "mention_saja"
    elif is_mention_and_short_word(text):
        return "mention_dan_kata_pendek"
    elif is_single_word(text):
        return "satu_kata_saja"
    elif is_dominantly_mention(text):
        return "mayoritas_mention"
    elif is_mention_and_short_word(text):
        return "mention_dan_kata_pendek"

    else:
        return None
    
    
# --- Muat dan proses data ---
# Ganti path file sesuai lokasi datamu
INPUT_FILE = "D:/kuliah/Skripsi/clean-code/final_results(2).csv"
OUTPUT_CLEAN = "D:/kuliah/Skripsi/data/datafixbersih/cleaned_comments_filtered(fr2).csv"
OUTPUT_REMOVED = "D:/kuliah/Skripsi/data/datafixbersih/log_removed_comments(fr2).csv"

# Baca file CSV
df = pd.read_csv(INPUT_FILE)
initial_count = len(df) # Hitung jumlah komentar awal

# buang baris dengan komentar kosong
df = df.dropna(subset=["T_komentar"])
processed_count = len(df) # Hitung jumlah setelah membuang baris kosong

# Lakukan preprocessing terhadap kolom komentar
df["T_komentar"] = df["T_komentar"].apply(preprocess_text)

# Tandai komentar yang perlu dibuang beserta alasannya
df["removal_reason"] = df["T_komentar"].apply(get_removal_reason)

# Bagi data menjadi dua: bersih dan dibuang
df_cleaned = df[df["removal_reason"].isna()].copy()      # yang tidak dibuang
df_removed = df[df["removal_reason"].notna()].copy()     # yang dibuang

# Simpan hasil ke file CSV
df_cleaned.drop(columns=["removal_reason"]).to_csv(OUTPUT_CLEAN, index=False)
df_removed.to_csv(OUTPUT_REMOVED, index=False)

# Ringkasan hasil
print(f"📖 Total komentar awal di file: {initial_count}")
print(f"⚙️ Total komentar diproses (setelah membuang baris kosong): {processed_count}")
print(f"✅ Total komentar bersih: {len(df_cleaned)} (disimpan di {OUTPUT_CLEAN})")
print(f"🗑️ Total komentar dibuang: {len(df_removed)} (disimpan di {OUTPUT_REMOVED})")


📖 Total komentar awal di file: 13150
⚙️ Total komentar diproses (setelah membuang baris kosong): 13150
✅ Total komentar bersih: 10412 (disimpan di D:/kuliah/Skripsi/data/datafixbersih/cleaned_comments_filtered(fr2).csv)
🗑️ Total komentar dibuang: 2738 (disimpan di D:/kuliah/Skripsi/data/datafixbersih/log_removed_comments(fr2).csv)


In [127]:
# youtube_commetn(22).csv

# pakai
import re
import string
import emoji
import unicodedata
import pandas as pd


# --- Fungsi bantu ---
def extract_links(text):
    return re.findall(r"https?://\S+", text)

# Mapping karakter non-Latin (Cyrillic) yang mirip huruf Latin
visual_map = {
    'А': 'A', 'В': 'B', 'С': 'C', 'Е': 'E', 'Н': 'H',
    'К': 'K', 'М': 'M', 'О': 'O', 'Р': 'P', 'Т': 'T',
    'Х': 'X', 'а': 'a', 'е': 'e', 'о': 'o', 'р': 'p',
    'с': 'c', 'у': 'y', 'х': 'x',
}

def merge_split_letters(text):
    """
    Gabungkan kembali huruf-huruf satuan yang dipisah spasi.
    Contoh: 'ᴡ ᴀ ᴋ ᴀ ᴛ ᴏ ᴛ ᴏ' → 'ᴡᴀᴋᴀᴛᴏᴛᴏ'
    """
    tokens = text.strip().split()
    merged = []
    temp = ""

    for token in tokens:
        if len(token) == 1 and unicodedata.category(token).startswith("L"):
            temp += token
        else:
            if temp:
                merged.append(temp)
                temp = ""
            merged.append(token)

    if temp:
        merged.append(temp)

    return " ".join(merged)


def preprocess_text(text):
    """
    Normalisasi teks komentar:
    - huruf kecil
    - link → LINK0, LINK1, ...
    - hapus timestamp (hh:mm)
    - hapus tanda baca ASCII (kecuali @ dan ?)
    - hapus karakter non-latin dan non-emoji dari tiap kata
    - rapikan spasi
    """
    text = str(text).lower()

    # Ganti semua link dengan token LINK0, LINK1, ...
    links = extract_links(text)
    for i, link in enumerate(links):
        text = text.replace(link, f"LINK{i}")

    # Hapus format jam (misalnya 11:35)
    text = re.sub(r"\b\d{1,2}:\d{2}\b", "", text)

    # Hapus tanda baca ASCII (kecuali @ dan ?)
    ascii_punctuation = ''.join(c for c in string.punctuation if c not in "@?")
    text = re.sub(rf"[{re.escape(ascii_punctuation)}]", " ", text)

    # Ganti karakter non-printable
    text = re.sub(r"[\u0000-\u001F\u007F-\u009F\u200B-\u200D]+", " ", text)

    def is_valid_text_char(char):
        # PERBAIKAN: Memastikan return False berada di dalam blok try-except
        try:
            name = unicodedata.name(char)
            # Hanya huruf/angka Latin atau gaya Latin matematika (bold/italic/fancy)
            if "LATIN" in name or "MATHEMATICAL" in name or "DIGIT" in name:
                return True
        except ValueError:
            # Jika karakter tidak punya nama (misalnya, beberapa simbol), anggap tidak valid
            return False
        # Jika punya nama tapi tidak cocok kriteria di atas, anggap tidak valid
        return False

    def clean_word(word):
        if word.lower().startswith("link"):
            return word

        chars_to_remove = {'♥', '❤', '💖', '💘', '💗', '💓', '💝'}  # simbol love
        result = ""

        if word.startswith("@"):
            # Pecah dengan regex: @username + sisa karakter
            # Pecah @username dari sisanya
            match = re.match(r"^@[\w\d_]+", word) # match @username
            if match:
                username = match.group()
                tail = word[len(username):]
            else:
                username = "@"
                tail = word[1:]
            
            # Bersihkan tail: buang simbol non-emoji (kecuali emoji tetap)
            # Sisakan hanya emoji dari tail
            tail_cleaned = ''.join(
                c for c in tail if c not in chars_to_remove and c in emoji.EMOJI_DATA
            )

            return f"{username} {tail_cleaned}".strip()

        for char in word:
            if char in chars_to_remove:
                continue
            if char in visual_map:
                result += visual_map[char]
            elif is_valid_text_char(char) or char in "@?" or char in emoji.EMOJI_DATA:
                result += char

        return result





    words = text.strip().split()
    words = [clean_word(word) for word in words if clean_word(word)]
    text = " ".join(words)
    text = merge_split_letters(text)  # ⬅️ Tambahkan ini untuk gabungkan huruf visual
    
    # PERBAIKAN: Memastikan fungsi selalu mengembalikan string
    return text



def is_only_emoji(text):
    return all(char in emoji.EMOJI_DATA for char in text.strip()) and len(text.strip()) > 0

def is_only_numbers(text):
    return re.fullmatch(r"\d+", text.strip()) is not None

def is_only_mentions(text):
    return bool(re.fullmatch(r"(@[^\s@]+[\s]*)+", text.strip()))

def is_single_word(text):
    return len(text.strip().split()) == 1

def is_only_mentions_and_emoji(text):
    words = text.strip().split()
    if not words:
        return False
    for word in words:
        if word.startswith("@"):
            continue
        elif all(char in emoji.EMOJI_DATA for char in word):
            continue
        else:
            return False
    return True

def is_only_mentions_links_and_emoji(text):
    words = text.strip().split()
    if not words:
        return False
    for word in words:
        if word.startswith("@"):
            continue
        elif re.match(r"https?://\S+", word):
            continue
        elif word.lower().startswith("link"):
            continue
        elif all(char in emoji.EMOJI_DATA for char in word):
            continue
        else:
            return False
    return True

def is_only_symbols_or_emoticons(text):
    """
    Buang komentar jika:
    - Tidak mengandung huruf Latin (a-z)
    - Terdiri dari karakter simbol, bentuk dekoratif, atau emoticon teks
    """
    text = text.strip()
    if not text:
        return False

    # Jika ada huruf Latin, anggap ada kata bermakna
    if re.search(r"[a-z]", text):
        return False

    # Jika panjang teks minimal dan semua bukan angka/alfabet → buang
    if len(text) == 0: return False # Hindari ZeroDivisionError
    non_alnum_ratio = sum(1 for c in text if not c.isalnum()) / len(text)
    return non_alnum_ratio > 0.6  # 60% lebih bukan huruf/angka

def is_mention_and_short_word(text):
    """
    Buang komentar jika hanya terdiri dari mention + satu kata pendek (panjang ≤ 3 huruf).
    Contoh: "@3 am", "@akun wkwk"
    """
    tokens = text.strip().split()
    if len(tokens) == 2 and tokens[0].startswith("@"):
        word = tokens[1]
        return len(word) <= 5
    return False

# def is_dominantly_mention(text):
#     """
#     Jika 70% atau lebih token adalah mention (@...), buang.
#     Kecuali jika ada kalimat panjang setelahnya.
#     """
    
#     tokens = text.strip().split()
#     if not tokens:
#         return False
#     mention_count = sum(1 for token in tokens if token.startswith("@"))
#     ratio = mention_count / len(tokens)
#     return ratio >= 0.66 and len(tokens) <= 5
def is_dominantly_mention(text):
    """
    Buang komentar jika:
    - ≥ 2 mention
    - dan sisa token tidak mengandung kata panjang bermakna
    - dan total token ≤ 6
    """
    tokens = text.strip().split()
    if not tokens:
        return False

    mention_tokens = [t for t in tokens if t.startswith("@")]
    non_mentions = [t for t in tokens if not t.startswith("@")]

    if len(mention_tokens) < 2 or len(tokens) > 6:
        return False

    # Jika semua non-mention pendek (≤4) dan bukan emoji
    semua_pendek = all(
        len(t) <= 4 and not all(char in emoji.EMOJI_DATA for char in t)
        for t in non_mentions
    )

    return semua_pendek






def get_removal_reason(text):
    # PERBAIKAN: Menambahkan pengecekan tipe data untuk keamanan
    if not isinstance(text, str) or not text.strip():
        return "kosong_setelah_pembersihan"
    elif is_only_mentions_links_and_emoji(text):
        return "mention_dan_link_atau_emoji_saja"
    elif is_only_mentions_and_emoji(text):
        return "mention_dan_emoji_saja"
    elif is_only_emoji(text):
        return "emoji_saja"
    elif is_only_symbols_or_emoticons(text):
        return "simbol_non_teks"
    elif is_only_numbers(text):
        return "angka_saja"
    elif is_only_mentions(text):
        return "mention_saja"
    elif is_mention_and_short_word(text):
        return "mention_dan_kata_pendek"
    elif is_single_word(text):
        return "satu_kata_saja"
    elif is_dominantly_mention(text):
        return "mayoritas_mention"
    elif is_mention_and_short_word(text):
        return "mention_dan_kata_pendek"

    else:
        return None
    
    
# --- Muat dan proses data ---
# Ganti path file sesuai lokasi datamu
INPUT_FILE = "D:/kuliah/Skripsi/clean-code/youtube_comments(22).csv"
OUTPUT_CLEAN = "D:/kuliah/Skripsi/data/datafixbersih/cleaned_comments_filtered(yc22).csv"
OUTPUT_REMOVED = "D:/kuliah/Skripsi/data/datafixbersih/log_removed_comments(yc22).csv"

# Baca file CSV
df = pd.read_csv(INPUT_FILE)
initial_count = len(df) # Hitung jumlah komentar awal

# buang baris dengan komentar kosong
df = df.dropna(subset=["T_komentar"])
processed_count = len(df) # Hitung jumlah setelah membuang baris kosong

# Lakukan preprocessing terhadap kolom komentar
df["T_komentar"] = df["T_komentar"].apply(preprocess_text)

# Tandai komentar yang perlu dibuang beserta alasannya
df["removal_reason"] = df["T_komentar"].apply(get_removal_reason)

# Bagi data menjadi dua: bersih dan dibuang
df_cleaned = df[df["removal_reason"].isna()].copy()      # yang tidak dibuang
df_removed = df[df["removal_reason"].notna()].copy()     # yang dibuang

# Simpan hasil ke file CSV
df_cleaned.drop(columns=["removal_reason"]).to_csv(OUTPUT_CLEAN, index=False)
df_removed.to_csv(OUTPUT_REMOVED, index=False)

# Ringkasan hasil
print(f"📖 Total komentar awal di file: {initial_count}")
print(f"⚙️ Total komentar diproses (setelah membuang baris kosong): {processed_count}")
print(f"✅ Total komentar bersih: {len(df_cleaned)} (disimpan di {OUTPUT_CLEAN})")
print(f"🗑️ Total komentar dibuang: {len(df_removed)} (disimpan di {OUTPUT_REMOVED})")

📖 Total komentar awal di file: 6998
⚙️ Total komentar diproses (setelah membuang baris kosong): 6998
✅ Total komentar bersih: 5975 (disimpan di D:/kuliah/Skripsi/data/datafixbersih/cleaned_comments_filtered(yc22).csv)
🗑️ Total komentar dibuang: 1023 (disimpan di D:/kuliah/Skripsi/data/datafixbersih/log_removed_comments(yc22).csv)


In [125]:
# final result(1).csv

# pakai
import re
import string
import emoji
import unicodedata
import pandas as pd


# --- Fungsi bantu ---
def extract_links(text):
    return re.findall(r"https?://\S+", text)

# Mapping karakter non-Latin (Cyrillic) yang mirip huruf Latin
visual_map = {
    'А': 'A', 'В': 'B', 'С': 'C', 'Е': 'E', 'Н': 'H',
    'К': 'K', 'М': 'M', 'О': 'O', 'Р': 'P', 'Т': 'T',
    'Х': 'X', 'а': 'a', 'е': 'e', 'о': 'o', 'р': 'p',
    'с': 'c', 'у': 'y', 'х': 'x',
}

def merge_split_letters(text):
    """
    Gabungkan kembali huruf-huruf satuan yang dipisah spasi.
    Contoh: 'ᴡ ᴀ ᴋ ᴀ ᴛ ᴏ ᴛ ᴏ' → 'ᴡᴀᴋᴀᴛᴏᴛᴏ'
    """
    tokens = text.strip().split()
    merged = []
    temp = ""

    for token in tokens:
        if len(token) == 1 and unicodedata.category(token).startswith("L"):
            temp += token
        else:
            if temp:
                merged.append(temp)
                temp = ""
            merged.append(token)

    if temp:
        merged.append(temp)

    return " ".join(merged)


def preprocess_text(text):
    """
    Normalisasi teks komentar:
    - huruf kecil
    - link → LINK0, LINK1, ...
    - hapus timestamp (hh:mm)
    - hapus tanda baca ASCII (kecuali @ dan ?)
    - hapus karakter non-latin dan non-emoji dari tiap kata
    - rapikan spasi
    """
    text = str(text).lower()

    # Ganti semua link dengan token LINK0, LINK1, ...
    links = extract_links(text)
    for i, link in enumerate(links):
        text = text.replace(link, f"LINK{i}")

    # Hapus format jam (misalnya 11:35)
    text = re.sub(r"\b\d{1,2}:\d{2}\b", "", text)

    # Hapus tanda baca ASCII (kecuali @ dan ?)
    ascii_punctuation = ''.join(c for c in string.punctuation if c not in "@?")
    text = re.sub(rf"[{re.escape(ascii_punctuation)}]", " ", text)

    # Ganti karakter non-printable
    text = re.sub(r"[\u0000-\u001F\u007F-\u009F\u200B-\u200D]+", " ", text)

    def is_valid_text_char(char):
        # PERBAIKAN: Memastikan return False berada di dalam blok try-except
        try:
            name = unicodedata.name(char)
            # Hanya huruf/angka Latin atau gaya Latin matematika (bold/italic/fancy)
            if "LATIN" in name or "MATHEMATICAL" in name or "DIGIT" in name:
                return True
        except ValueError:
            # Jika karakter tidak punya nama (misalnya, beberapa simbol), anggap tidak valid
            return False
        # Jika punya nama tapi tidak cocok kriteria di atas, anggap tidak valid
        return False

    def clean_word(word):
        if word.lower().startswith("link"):
            return word

        chars_to_remove = {'♥', '❤', '💖', '💘', '💗', '💓', '💝'}  # simbol love
        result = ""

        if word.startswith("@"):
            # Pecah dengan regex: @username + sisa karakter
            # Pecah @username dari sisanya
            match = re.match(r"^@[\w\d_]+", word) # match @username
            if match:
                username = match.group()
                tail = word[len(username):]
            else:
                username = "@"
                tail = word[1:]
            
            # Bersihkan tail: buang simbol non-emoji (kecuali emoji tetap)
            # Sisakan hanya emoji dari tail
            tail_cleaned = ''.join(
                c for c in tail if c not in chars_to_remove and c in emoji.EMOJI_DATA
            )

            return f"{username} {tail_cleaned}".strip()

        for char in word:
            if char in chars_to_remove:
                continue
            if char in visual_map:
                result += visual_map[char]
            elif is_valid_text_char(char) or char in "@?" or char in emoji.EMOJI_DATA:
                result += char

        return result





    words = text.strip().split()
    words = [clean_word(word) for word in words if clean_word(word)]
    text = " ".join(words)
    text = merge_split_letters(text)  # ⬅️ Tambahkan ini untuk gabungkan huruf visual
    
    # PERBAIKAN: Memastikan fungsi selalu mengembalikan string
    return text



def is_only_emoji(text):
    return all(char in emoji.EMOJI_DATA for char in text.strip()) and len(text.strip()) > 0

def is_only_numbers(text):
    return re.fullmatch(r"\d+", text.strip()) is not None

def is_only_mentions(text):
    return bool(re.fullmatch(r"(@[^\s@]+[\s]*)+", text.strip()))

def is_single_word(text):
    return len(text.strip().split()) == 1

def is_only_mentions_and_emoji(text):
    words = text.strip().split()
    if not words:
        return False
    for word in words:
        if word.startswith("@"):
            continue
        elif all(char in emoji.EMOJI_DATA for char in word):
            continue
        else:
            return False
    return True

def is_only_mentions_links_and_emoji(text):
    words = text.strip().split()
    if not words:
        return False
    for word in words:
        if word.startswith("@"):
            continue
        elif re.match(r"https?://\S+", word):
            continue
        elif word.lower().startswith("link"):
            continue
        elif all(char in emoji.EMOJI_DATA for char in word):
            continue
        else:
            return False
    return True

def is_only_symbols_or_emoticons(text):
    """
    Buang komentar jika:
    - Tidak mengandung huruf Latin (a-z)
    - Terdiri dari karakter simbol, bentuk dekoratif, atau emoticon teks
    """
    text = text.strip()
    if not text:
        return False

    # Jika ada huruf Latin, anggap ada kata bermakna
    if re.search(r"[a-z]", text):
        return False

    # Jika panjang teks minimal dan semua bukan angka/alfabet → buang
    if len(text) == 0: return False # Hindari ZeroDivisionError
    non_alnum_ratio = sum(1 for c in text if not c.isalnum()) / len(text)
    return non_alnum_ratio > 0.6  # 60% lebih bukan huruf/angka

def is_mention_and_short_word(text):
    """
    Buang komentar jika hanya terdiri dari mention + satu kata pendek (panjang ≤ 3 huruf).
    Contoh: "@3 am", "@akun wkwk"
    """
    tokens = text.strip().split()
    if len(tokens) == 2 and tokens[0].startswith("@"):
        word = tokens[1]
        return len(word) <= 5
    return False

# def is_dominantly_mention(text):
#     """
#     Jika 70% atau lebih token adalah mention (@...), buang.
#     Kecuali jika ada kalimat panjang setelahnya.
#     """
    
#     tokens = text.strip().split()
#     if not tokens:
#         return False
#     mention_count = sum(1 for token in tokens if token.startswith("@"))
#     ratio = mention_count / len(tokens)
#     return ratio >= 0.66 and len(tokens) <= 5
def is_dominantly_mention(text):
    """
    Buang komentar jika:
    - ≥ 2 mention
    - dan sisa token tidak mengandung kata panjang bermakna
    - dan total token ≤ 6
    """
    tokens = text.strip().split()
    if not tokens:
        return False

    mention_tokens = [t for t in tokens if t.startswith("@")]
    non_mentions = [t for t in tokens if not t.startswith("@")]

    if len(mention_tokens) < 2 or len(tokens) > 6:
        return False

    # Jika semua non-mention pendek (≤4) dan bukan emoji
    semua_pendek = all(
        len(t) <= 4 and not all(char in emoji.EMOJI_DATA for char in t)
        for t in non_mentions
    )

    return semua_pendek






def get_removal_reason(text):
    # PERBAIKAN: Menambahkan pengecekan tipe data untuk keamanan
    if not isinstance(text, str) or not text.strip():
        return "kosong_setelah_pembersihan"
    elif is_only_mentions_links_and_emoji(text):
        return "mention_dan_link_atau_emoji_saja"
    elif is_only_mentions_and_emoji(text):
        return "mention_dan_emoji_saja"
    elif is_only_emoji(text):
        return "emoji_saja"
    elif is_only_symbols_or_emoticons(text):
        return "simbol_non_teks"
    elif is_only_numbers(text):
        return "angka_saja"
    elif is_only_mentions(text):
        return "mention_saja"
    elif is_mention_and_short_word(text):
        return "mention_dan_kata_pendek"
    elif is_single_word(text):
        return "satu_kata_saja"
    elif is_dominantly_mention(text):
        return "mayoritas_mention"
    elif is_mention_and_short_word(text):
        return "mention_dan_kata_pendek"

    else:
        return None
    
    
# --- Muat dan proses data ---

# Ganti path file sesuai lokasi datamu
INPUT_FILE = "D:/kuliah/Skripsi/clean-code/final_results(1).csv"
OUTPUT_CLEAN = "D:/kuliah/Skripsi/data/datafixbersih/cleaned_comments_filtered(fr1).csv"
OUTPUT_REMOVED = "D:/kuliah/Skripsi/data/datafixbersih/log_removed_comments(fr1).csv"

# Baca file CSV
df = pd.read_csv(INPUT_FILE)
initial_count = len(df) # Hitung jumlah komentar awal

# buang baris dengan komentar kosong
df = df.dropna(subset=["T_komentar"])
processed_count = len(df) # Hitung jumlah setelah membuang baris kosong

# Lakukan preprocessing terhadap kolom komentar
df["T_komentar"] = df["T_komentar"].apply(preprocess_text)

# Tandai komentar yang perlu dibuang beserta alasannya
df["removal_reason"] = df["T_komentar"].apply(get_removal_reason)

# Bagi data menjadi dua: bersih dan dibuang
df_cleaned = df[df["removal_reason"].isna()].copy()      # yang tidak dibuang
df_removed = df[df["removal_reason"].notna()].copy()     # yang dibuang

# Simpan hasil ke file CSV
df_cleaned.drop(columns=["removal_reason"]).to_csv(OUTPUT_CLEAN, index=False)
df_removed.to_csv(OUTPUT_REMOVED, index=False)

# Ringkasan hasil
print(f"📖 Total komentar awal di file: {initial_count}")
print(f"⚙️ Total komentar diproses (setelah membuang baris kosong): {processed_count}")
print(f"✅ Total komentar bersih: {len(df_cleaned)} (disimpan di {OUTPUT_CLEAN})")
print(f"🗑️ Total komentar dibuang: {len(df_removed)} (disimpan di {OUTPUT_REMOVED})")

📖 Total komentar awal di file: 30190
⚙️ Total komentar diproses (setelah membuang baris kosong): 30190
✅ Total komentar bersih: 20349 (disimpan di D:/kuliah/Skripsi/data/datafixbersih/cleaned_comments_filtered(fr1).csv)
🗑️ Total komentar dibuang: 9841 (disimpan di D:/kuliah/Skripsi/data/datafixbersih/log_removed_comments(fr1).csv)


In [130]:
# cleancommet(nonjudi).csv

# pakai
import re
import string
import emoji
import unicodedata
import pandas as pd


# --- Fungsi bantu ---
def extract_links(text):
    return re.findall(r"https?://\S+", text)

# Mapping karakter non-Latin (Cyrillic) yang mirip huruf Latin
visual_map = {
    'А': 'A', 'В': 'B', 'С': 'C', 'Е': 'E', 'Н': 'H',
    'К': 'K', 'М': 'M', 'О': 'O', 'Р': 'P', 'Т': 'T',
    'Х': 'X', 'а': 'a', 'е': 'e', 'о': 'o', 'р': 'p',
    'с': 'c', 'у': 'y', 'х': 'x',
}

def merge_split_letters(text):
    """
    Gabungkan kembali huruf-huruf satuan yang dipisah spasi.
    Contoh: 'ᴡ ᴀ ᴋ ᴀ ᴛ ᴏ ᴛ ᴏ' → 'ᴡᴀᴋᴀᴛᴏᴛᴏ'
    """
    tokens = text.strip().split()
    merged = []
    temp = ""

    for token in tokens:
        if len(token) == 1 and unicodedata.category(token).startswith("L"):
            temp += token
        else:
            if temp:
                merged.append(temp)
                temp = ""
            merged.append(token)

    if temp:
        merged.append(temp)

    return " ".join(merged)


def preprocess_text(text):
    """
    Normalisasi teks komentar:
    - huruf kecil
    - link → LINK0, LINK1, ...
    - hapus timestamp (hh:mm)
    - hapus tanda baca ASCII (kecuali @ dan ?)
    - hapus karakter non-latin dan non-emoji dari tiap kata
    - rapikan spasi
    """
    text = str(text).lower()

    # Ganti semua link dengan token LINK0, LINK1, ...
    links = extract_links(text)
    for i, link in enumerate(links):
        text = text.replace(link, f"LINK{i}")

    # Hapus format jam (misalnya 11:35)
    text = re.sub(r"\b\d{1,2}:\d{2}\b", "", text)

    # Hapus tanda baca ASCII (kecuali @ dan ?)
    ascii_punctuation = ''.join(c for c in string.punctuation if c not in "@?")
    text = re.sub(rf"[{re.escape(ascii_punctuation)}]", " ", text)

    # Ganti karakter non-printable
    text = re.sub(r"[\u0000-\u001F\u007F-\u009F\u200B-\u200D]+", " ", text)

    def is_valid_text_char(char):
        # PERBAIKAN: Memastikan return False berada di dalam blok try-except
        try:
            name = unicodedata.name(char)
            # Hanya huruf/angka Latin atau gaya Latin matematika (bold/italic/fancy)
            if "LATIN" in name or "MATHEMATICAL" in name or "DIGIT" in name:
                return True
        except ValueError:
            # Jika karakter tidak punya nama (misalnya, beberapa simbol), anggap tidak valid
            return False
        # Jika punya nama tapi tidak cocok kriteria di atas, anggap tidak valid
        return False

    def clean_word(word):
        if word.lower().startswith("link"):
            return word

        chars_to_remove = {'♥', '❤', '💖', '💘', '💗', '💓', '💝'}  # simbol love
        result = ""

        if word.startswith("@"):
            # Pecah dengan regex: @username + sisa karakter
            # Pecah @username dari sisanya
            match = re.match(r"^@[\w\d_]+", word) # match @username
            if match:
                username = match.group()
                tail = word[len(username):]
            else:
                username = "@"
                tail = word[1:]
            
            # Bersihkan tail: buang simbol non-emoji (kecuali emoji tetap)
            # Sisakan hanya emoji dari tail
            tail_cleaned = ''.join(
                c for c in tail if c not in chars_to_remove and c in emoji.EMOJI_DATA
            )

            return f"{username} {tail_cleaned}".strip()

        for char in word:
            if char in chars_to_remove:
                continue
            if char in visual_map:
                result += visual_map[char]
            elif is_valid_text_char(char) or char in "@?" or char in emoji.EMOJI_DATA:
                result += char

        return result





    words = text.strip().split()
    words = [clean_word(word) for word in words if clean_word(word)]
    text = " ".join(words)
    text = merge_split_letters(text)  # ⬅️ Tambahkan ini untuk gabungkan huruf visual
    
    # PERBAIKAN: Memastikan fungsi selalu mengembalikan string
    return text



def is_only_emoji(text):
    return all(char in emoji.EMOJI_DATA for char in text.strip()) and len(text.strip()) > 0

def is_only_numbers(text):
    return re.fullmatch(r"\d+", text.strip()) is not None

def is_only_mentions(text):
    return bool(re.fullmatch(r"(@[^\s@]+[\s]*)+", text.strip()))

def is_single_word(text):
    return len(text.strip().split()) == 1

def is_only_mentions_and_emoji(text):
    words = text.strip().split()
    if not words:
        return False
    for word in words:
        if word.startswith("@"):
            continue
        elif all(char in emoji.EMOJI_DATA for char in word):
            continue
        else:
            return False
    return True

def is_only_mentions_links_and_emoji(text):
    words = text.strip().split()
    if not words:
        return False
    for word in words:
        if word.startswith("@"):
            continue
        elif re.match(r"https?://\S+", word):
            continue
        elif word.lower().startswith("link"):
            continue
        elif all(char in emoji.EMOJI_DATA for char in word):
            continue
        else:
            return False
    return True

def is_only_symbols_or_emoticons(text):
    """
    Buang komentar jika:
    - Tidak mengandung huruf Latin (a-z)
    - Terdiri dari karakter simbol, bentuk dekoratif, atau emoticon teks
    """
    text = text.strip()
    if not text:
        return False

    # Jika ada huruf Latin, anggap ada kata bermakna
    if re.search(r"[a-z]", text):
        return False

    # Jika panjang teks minimal dan semua bukan angka/alfabet → buang
    if len(text) == 0: return False # Hindari ZeroDivisionError
    non_alnum_ratio = sum(1 for c in text if not c.isalnum()) / len(text)
    return non_alnum_ratio > 0.6  # 60% lebih bukan huruf/angka

def is_mention_and_short_word(text):
    """
    Buang komentar jika hanya terdiri dari mention + satu kata pendek (panjang ≤ 3 huruf).
    Contoh: "@3 am", "@akun wkwk"
    """
    tokens = text.strip().split()
    if len(tokens) == 2 and tokens[0].startswith("@"):
        word = tokens[1]
        return len(word) <= 5
    return False

# def is_dominantly_mention(text):
#     """
#     Jika 70% atau lebih token adalah mention (@...), buang.
#     Kecuali jika ada kalimat panjang setelahnya.
#     """
    
#     tokens = text.strip().split()
#     if not tokens:
#         return False
#     mention_count = sum(1 for token in tokens if token.startswith("@"))
#     ratio = mention_count / len(tokens)
#     return ratio >= 0.66 and len(tokens) <= 5
def is_dominantly_mention(text):
    """
    Buang komentar jika:
    - ≥ 2 mention
    - dan sisa token tidak mengandung kata panjang bermakna
    - dan total token ≤ 6
    """
    tokens = text.strip().split()
    if not tokens:
        return False

    mention_tokens = [t for t in tokens if t.startswith("@")]
    non_mentions = [t for t in tokens if not t.startswith("@")]

    if len(mention_tokens) < 2 or len(tokens) > 6:
        return False

    # Jika semua non-mention pendek (≤4) dan bukan emoji
    semua_pendek = all(
        len(t) <= 4 and not all(char in emoji.EMOJI_DATA for char in t)
        for t in non_mentions
    )

    return semua_pendek






def get_removal_reason(text):
    # PERBAIKAN: Menambahkan pengecekan tipe data untuk keamanan
    if not isinstance(text, str) or not text.strip():
        return "kosong_setelah_pembersihan"
    elif is_only_mentions_links_and_emoji(text):
        return "mention_dan_link_atau_emoji_saja"
    elif is_only_mentions_and_emoji(text):
        return "mention_dan_emoji_saja"
    elif is_only_emoji(text):
        return "emoji_saja"
    elif is_only_symbols_or_emoticons(text):
        return "simbol_non_teks"
    elif is_only_numbers(text):
        return "angka_saja"
    elif is_only_mentions(text):
        return "mention_saja"
    elif is_mention_and_short_word(text):
        return "mention_dan_kata_pendek"
    elif is_single_word(text):
        return "satu_kata_saja"
    elif is_dominantly_mention(text):
        return "mayoritas_mention"
    elif is_mention_and_short_word(text):
        return "mention_dan_kata_pendek"

    else:
        return None
    
    
# --- Muat dan proses data ---

# Ganti path file sesuai lokasi datamu
INPUT_FILE = "cleaned_comments (1)(nojudi).csv"
OUTPUT_CLEAN = "D:/kuliah/Skripsi/data/datafixbersih/cleaned_comments_filtered(cc1).csv"
OUTPUT_REMOVED = "D:/kuliah/Skripsi/data/datafixbersih/log_removed_comments(cc1).csv"

# Baca file CSV
df = pd.read_csv(INPUT_FILE)
initial_count = len(df) # Hitung jumlah komentar awal

# buang baris dengan komentar kosong
df = df.dropna(subset=["textOriginal"])
processed_count = len(df) # Hitung jumlah setelah membuang baris kosong

# Lakukan preprocessing terhadap kolom komentar
df["textOriginal"] = df["textOriginal"].apply(preprocess_text)

# Tandai komentar yang perlu dibuang beserta alasannya
df["removal_reason"] = df["textOriginal"].apply(get_removal_reason)

# Bagi data menjadi dua: bersih dan dibuang
df_cleaned = df[df["removal_reason"].isna()].copy()      # yang tidak dibuang
df_removed = df[df["removal_reason"].notna()].copy()     # yang dibuang

# Simpan hasil ke file CSV
df_cleaned.drop(columns=["removal_reason"]).to_csv(OUTPUT_CLEAN, index=False)
df_removed.to_csv(OUTPUT_REMOVED, index=False)

# Ringkasan hasil
print(f"📖 Total komentar awal di file: {initial_count}")
print(f"⚙️ Total komentar diproses (setelah membuang baris kosong): {processed_count}")
print(f"✅ Total komentar bersih: {len(df_cleaned)} (disimpan di {OUTPUT_CLEAN})")
print(f"🗑️ Total komentar dibuang: {len(df_removed)} (disimpan di {OUTPUT_REMOVED})")

📖 Total komentar awal di file: 10939
⚙️ Total komentar diproses (setelah membuang baris kosong): 10938
✅ Total komentar bersih: 10161 (disimpan di D:/kuliah/Skripsi/data/datafixbersih/cleaned_comments_filtered(cc1).csv)
🗑️ Total komentar dibuang: 777 (disimpan di D:/kuliah/Skripsi/data/datafixbersih/log_removed_comments(cc1).csv)


## Versi 1.1

In [31]:
import pandas as pd
import re
import string
import emoji

# --- Fungsi bantu ---

def extract_links(text):
    # Mengambil semua link dalam teks
    return re.findall(r"https?://\S+", text)

def preprocess_text(text):
    """
    Melakukan normalisasi teks komentar:
    - huruf kecil
    - link diganti token LINK0, LINK1, dst
    - hapus timestamp seperti 11:35
    - hapus tanda baca (kecuali @ dan ?)
    - rapikan spasi
    """
    text = str(text).lower()
    
    # Simpan link asli untuk ditandai dengan token
    links = extract_links(text)

    # Ganti semua link dengan token LINK0, LINK1, ...
    for i, link in enumerate(links):
        text = text.replace(link, f"LINK{i}")

    # Hapus format jam (misalnya 11:35)
    text = re.sub(r"\b\d{1,2}:\d{2}\b", "", text)

    # Hapus semua tanda baca kecuali @ dan ?
    punctuation = string.punctuation.replace("@", "").replace("?", "")
    text = re.sub(rf"[{re.escape(punctuation)}]", " ", text)

    # Rapikan spasi yang berlebihan
    text = re.sub(r"\s+", " ", text).strip()
    return text

def is_only_emoji(text):
    # Deteksi jika seluruh teks hanya emoji
    return all(char in emoji.EMOJI_DATA for char in text.strip()) and len(text.strip()) > 0

def is_only_numbers(text):
    # Deteksi jika teks hanya angka
    return re.fullmatch(r"\d+", text.strip()) is not None

def is_only_mentions(text):
    # Deteksi jika teks hanya berisi mention seperti @user @admin
    return bool(re.fullmatch(r"(@[^\s@]+[\s]*)+", text.strip()))

def is_single_word(text):
    # Deteksi jika hanya terdiri dari 1 kata
    return len(text.strip().split()) == 1

def is_only_mentions_and_emoji(text):
    """
    Deteksi komentar yang hanya berisi kombinasi:
    - mention
    - emoji
    Tidak ada kata bermakna lain
    """
    words = text.strip().split()
    if not words:
        return False

    for word in words:
        if word.startswith("@"):
            continue
        elif all(char in emoji.EMOJI_DATA for char in word):
            continue
        else:
            return False
    return True

def is_only_mentions_links_and_emoji(text):
    """
    Deteksi komentar yang hanya berisi kombinasi dari:
    - mention
    - emoji
    - link (baik masih asli maupun sudah jadi LINK0, LINK1, ...)
    """
    words = text.strip().split()
    if not words:
        return False

    for word in words:
        if word.startswith("@"):
            continue
        elif re.match(r"https?://\S+", word):  # link asli (seandainya belum ditoken)
            continue
        elif word.lower().startswith("link"):  # token link (LINK0, LINK1, ...)
            continue
        elif all(char in emoji.EMOJI_DATA for char in word):
            continue
        else:
            return False
    return True

def get_removal_reason(text):
    """
    Memberikan alasan kenapa komentar dibuang
    Jika tidak memenuhi kriteria pembuangan → return None
    """
    if is_only_mentions_links_and_emoji(text):
        return "mention_dan_link_atau_emoji_saja"
    elif is_only_mentions_and_emoji(text):
        return "mention_dan_emoji_saja"
    elif is_only_emoji(text):
        return "emoji_saja"
    elif is_only_numbers(text):
        return "angka_saja"
    elif is_only_mentions(text):
        return "mention_saja"
    elif is_single_word(text):
        return "satu_kata_saja"
    else:
        return None

# --- Muat dan proses data ---

# Ganti path file sesuai lokasi datamu
INPUT_FILE = "D:/kuliah/Skripsi/clean-code/final_results(1).csv"
OUTPUT_CLEAN = "cleaned_comments_filtered(fr1b).csv"
OUTPUT_REMOVED = "log_removed_comments(fr1b).csv"

# Baca file CSV dan buang baris dengan komentar kosong
df = pd.read_csv(INPUT_FILE)
df = df.dropna(subset=["T_komentar"])

# Lakukan preprocessing terhadap kolom komentar
df["T_komentar"] = df["T_komentar"].apply(preprocess_text)

# Tandai komentar yang perlu dibuang beserta alasannya
df["removal_reason"] = df["T_komentar"].apply(get_removal_reason)

# Bagi data menjadi dua: bersih dan dibuang
df_cleaned = df[df["removal_reason"].isna()].copy()      # yang tidak dibuang
df_removed = df[df["removal_reason"].notna()].copy()     # yang dibuang

# Simpan hasil ke file CSV
df_cleaned.drop(columns=["removal_reason"]).to_csv(OUTPUT_CLEAN, index=False)
df_removed.to_csv(OUTPUT_REMOVED, index=False)

# Ringkasan hasil
print(f"✅ Total komentar bersih: {len(df_cleaned)} (disimpan di {OUTPUT_CLEAN})")
print(f"🗑️ Total komentar dibuang: {len(df_removed)} (disimpan di {OUTPUT_REMOVED})")


✅ Total komentar bersih: 20612 (disimpan di cleaned_comments_filtered(fr1b).csv)
🗑️ Total komentar dibuang: 9578 (disimpan di log_removed_comments(fr1b).csv)


## Versi 2

In [23]:
import re
import string
import emoji
import pandas as pd

# --- Fungsi bantu ---
def extract_links(text):
    # Ekstrak semua link dari teks
    return re.findall(r"https?://\S+", text)

def preprocess_text(text):
    text = str(text).lower()

    # Simpan dan ganti semua link dengan token LINK0, LINK1, dst.
    links = extract_links(text)
    for i, link in enumerate(links):
        text = text.replace(link, f"LINK{i}")

    # Hapus timestamp seperti 07:32
    text = re.sub(r"\b\d{1,2}:\d{2}\b", "", text)

    # Hapus tanda baca kecuali @ dan ?
    punctuation = string.punctuation.replace("@", "").replace("?", "")
    text = re.sub(rf"[{re.escape(punctuation)}]", " ", text)

    # Hapus spasi berlebih
    text = re.sub(r"\s+", " ", text).strip()
    return text

def is_only_emoji(text):
    return all(char in emoji.EMOJI_DATA for char in text.strip()) and len(text.strip()) > 0

def is_only_numbers(text):
    return re.fullmatch(r"\d+", text.strip()) is not None

def is_single_word(text):
    return len(text.strip().split()) == 1

def is_only_mentions(text):
    words = text.strip().split()
    if not words:
        return False

    # Hitung mention dan simpan kata non-mention
    mention_count = sum(1 for word in words if word.startswith("@"))
    non_mention_words = [word for word in words if not word.startswith("@")]

    # Semua mention → buang
    if len(non_mention_words) == 0:
        return True

    # Kalau semua sisa katanya pendek dan alfabet → juga buang
    if all(len(w) <= 10 and w.isalpha() for w in non_mention_words):
        return True

    return False

def is_only_mentions_and_emoji(text):
    words = text.strip().split()
    if not words:
        return False

    meaningful_word_found = False

    for word in words:
        if word.startswith("@"):
            continue
        elif all(char in emoji.EMOJI_DATA for char in word):
            continue
        elif word.lower().startswith("link"):
            continue
        elif word.isalpha():
            if len(word) > 2:  # Kata bermakna seperti 'mampir', 'dong', dst.
                meaningful_word_found = True
            else:
                continue  # kata pendek → abaikan
        else:
            return False  # karakter lain (angka/simbol) → jangan dibuang

    # Jika tidak ada kata bermakna → buang
    return not meaningful_word_found



def is_only_mentions_links_and_emoji(text):
    words = text.strip().split()
    if not words:
        return False
    for word in words:
        if word.startswith("@"):
            continue
        elif word.lower().startswith("link"):  # token LINK0, LINK1, dll
            continue
        elif re.match(r"https?://\S+", word):  # link asli
            continue
        elif all(char in emoji.EMOJI_DATA for char in word):
            continue
        elif word.isalpha():  # Ada kata bermakna → jangan buang
            return False
        else:
            return False
    return True

def get_removal_reason(text):
    if is_only_mentions_links_and_emoji(text):
        return "mention_dan_link_atau_emoji_saja"
    elif is_only_mentions_and_emoji(text):
        return "mention_dan_emoji_saja"
    elif is_only_emoji(text):
        return "emoji_saja"
    elif is_only_numbers(text):
        return "angka_saja"
    elif is_only_mentions(text):
        return "mention_saja"
    elif is_single_word(text):
        return "satu_kata_saja"
    else:
        return None

# --- Input manual dari user untuk diuji ---
sample_comments = [
    "mampir dong",
]

# --- Proses dan tampilkan hasil ---
results = []
for comment in sample_comments:
    cleaned = preprocess_text(comment)
    reason = get_removal_reason(cleaned)
    status = "✅ dipertahankan" if reason is None else f"🗑️ dibuang ({reason})"
    results.append({
        "Komentar Asli": comment,
        "Setelah Dibersihkan": cleaned,
        "Status": status
    })

# Tampilkan hasil sebagai tabel
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))


Komentar Asli Setelah Dibersihkan                    Status
  mampir dong         mampir dong 🗑️ dibuang (mention_saja)


## versi 3

In [28]:
import re
import string
import emoji
import pandas as pd

# --- Fungsi bantu ---
def extract_links(text):
    return re.findall(r"https?://\S+", text)

def preprocess_text(text):
    text = str(text).lower()
    links = extract_links(text)
    for i, link in enumerate(links):
        text = text.replace(link, f"LINK{i}")
    text = re.sub(r"\b\d{1,2}:\d{2}\b", "", text)
    punctuation = string.punctuation.replace("@", "").replace("?", "")
    text = re.sub(rf"[{re.escape(punctuation)}]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def is_only_emoji(text):
    return all(char in emoji.EMOJI_DATA for char in text.strip()) and len(text.strip()) > 0

def is_only_numbers(text):
    return re.fullmatch(r"\d+", text.strip()) is not None

def is_single_word(text):
    return len(text.strip().split()) == 1

def is_only_mentions(text):
    words = text.strip().split()
    if not words:
        return False
    mention_count = sum(1 for word in words if word.startswith("@"))
    non_mention_words = [word for word in words if not word.startswith("@")]
    if len(non_mention_words) == 0:
        return True
    if all(len(w) <= 10 and w.isalpha() for w in non_mention_words):
        return True
    return False

def is_only_mentions_and_emoji(text):
    words = text.strip().split()
    if not words:
        return False
    meaningful_word_found = False
    for word in words:
        if word.startswith("@"):
            continue
        elif word.lower().startswith("link"):
            continue
        elif all(char in emoji.EMOJI_DATA for char in word):
            continue
        elif word.isalpha():
            if len(word) > 2:
                meaningful_word_found = True
            else:
                continue
        else:
            return False
    return not meaningful_word_found

def is_only_mentions_links_and_emoji(text):
    words = text.strip().split()
    if not words:
        return False
    for word in words:
        if word.startswith("@"):
            continue
        elif word.lower().startswith("link"):
            continue
        elif re.match(r"https?://\S+", word):
            continue
        elif all(char in emoji.EMOJI_DATA for char in word):
            continue
        elif word.isalpha():
            return False
        else:
            return False
    return True

def get_removal_reason(text):
    if is_only_mentions_links_and_emoji(text):
        return "mention_dan_link_atau_emoji_saja"
    elif is_only_mentions_and_emoji(text):
        return "mention_dan_emoji_saja"
    elif is_only_emoji(text):
        return "emoji_saja"
    elif is_only_numbers(text):
        return "angka_saja"
    elif is_only_mentions(text):
        return "mention_saja"
    elif is_single_word(text):
        return "satu_kata_saja"
    else:
        return None

# --- Komentar uji coba manual ---
sample_comments = [
    "@m akwoakwow",
    "@i n i s i a l r @apedzpocoong88",
    "@netral23 @tuan muda",
    "@anak baik🔞"
]

# --- Proses dan tampilkan hasil ---
results = []
for comment in sample_comments:
    cleaned = preprocess_text(comment)
    reason = get_removal_reason(cleaned)
    status = "✅ dipertahankan" if reason is None else f"🗑️ dibuang ({reason})"
    results.append({
        "Komentar Asli": comment,
        "Setelah Dibersihkan": cleaned,
        "Status": status
    })

df = pd.DataFrame(results)
print(df.to_string(index=False))


                   Komentar Asli              Setelah Dibersihkan                              Status
                    @m akwoakwow                     @m akwoakwow           🗑️ dibuang (mention_saja)
@i n i s i a l r @apedzpocoong88 @i n i s i a l r @apedzpocoong88 🗑️ dibuang (mention_dan_emoji_saja)
            @netral23 @tuan muda             @netral23 @tuan muda           🗑️ dibuang (mention_saja)
                     @anak baik🔞                      @anak baik🔞                     ✅ dipertahankan


In [29]:
import pandas as pd
import re
import string
import emoji

# --- Fungsi bantu ---
def extract_links(text):
    return re.findall(r"https?://\S+", text)

def preprocess_text(text):
    text = str(text).lower()

    # Ganti semua link jadi LINK0, LINK1, dst.
    links = extract_links(text)
    for i, link in enumerate(links):
        text = text.replace(link, f"LINK{i}")

    # Hapus timestamp
    text = re.sub(r"\b\d{1,2}:\d{2}\b", "", text)

    # Hapus tanda baca kecuali @ dan ?
    punctuation = string.punctuation.replace("@", "").replace("?", "")
    text = re.sub(rf"[{re.escape(punctuation)}]", " ", text)

    # Hapus spasi berlebih
    text = re.sub(r"\s+", " ", text).strip()
    return text

def is_only_emoji(text):
    return all(char in emoji.EMOJI_DATA for char in text.strip()) and len(text.strip()) > 0

def is_only_numbers(text):
    return re.fullmatch(r"\d+", text.strip()) is not None

def is_single_word(text):
    return len(text.strip().split()) == 1

def is_only_mentions(text):
    words = text.strip().split()
    if not words:
        return False

    mention_count = sum(1 for word in words if word.startswith("@"))
    non_mention_words = [word for word in words if not word.startswith("@")]

    if len(non_mention_words) == 0:
        return True

    if all(len(w) <= 10 and w.isalpha() for w in non_mention_words):
        return True

    return False

def is_only_mentions_and_emoji(text):
    """
    Buang komentar yang hanya terdiri dari:
    - mention (termasuk @ + nama panjang)
    - emoji
    - kata acak pendek (≤2 huruf)
    - token LINK
    Pertahankan jika ada minimal 1 kata bermakna (panjang > 2 huruf)
    """
    words = text.strip().split()
    if not words:
        return False

    meaningful_word_found = False

    for word in words:
        if word.startswith("@"):
            continue
        elif word.lower().startswith("link"):
            continue
        elif all(char in emoji.EMOJI_DATA for char in word):
            continue
        elif word.isalpha():
            if len(word) > 2:
                meaningful_word_found = True
            else:
                continue  # kata pendek seperti 'u', 't' → abaikan
        else:
            return False

    return not meaningful_word_found

def is_only_mentions_links_and_emoji(text):
    words = text.strip().split()
    if not words:
        return False

    for word in words:
        if word.startswith("@"):
            continue
        elif word.lower().startswith("link"):
            continue
        elif re.match(r"https?://\S+", word):
            continue
        elif all(char in emoji.EMOJI_DATA for char in word):
            continue
        elif word.isalpha():
            return False
        else:
            return False
    return True

def get_removal_reason(text):
    if is_only_mentions_links_and_emoji(text):
        return "mention_dan_link_atau_emoji_saja"
    elif is_only_mentions_and_emoji(text):
        return "mention_dan_emoji_saja"
    elif is_only_emoji(text):
        return "emoji_saja"
    elif is_only_numbers(text):
        return "angka_saja"
    elif is_only_mentions(text):
        return "mention_saja"
    elif is_single_word(text):
        return "satu_kata_saja"
    else:
        return None

# --- Muat dan proses data ---
INPUT_FILE = "D:/kuliah/Skripsi/clean-code/final_results(1).csv"
OUTPUT_CLEAN = "cleaned_comments_filtered(fr1a).csv"
OUTPUT_REMOVED = "log_removed_comments(fr1a).csv"

# Baca file dan buang baris kosong
df = pd.read_csv(INPUT_FILE)
df = df.dropna(subset=["T_komentar"])

# Preprocessing
df["T_komentar"] = df["T_komentar"].apply(preprocess_text)

# Tandai komentar yang dibuang dan alasannya
df["removal_reason"] = df["T_komentar"].apply(get_removal_reason)

# Pisahkan
df_cleaned = df[df["removal_reason"].isna()].copy()
df_removed = df[df["removal_reason"].notna()].copy()

# Simpan ke file
df_cleaned.drop(columns=["removal_reason"]).to_csv(OUTPUT_CLEAN, index=False)
df_removed.to_csv(OUTPUT_REMOVED, index=False)

# Ringkasan
print(f"✅ Total komentar bersih: {len(df_cleaned)} (disimpan di {OUTPUT_CLEAN})")
print(f"🗑️ Total komentar dibuang: {len(df_removed)} (disimpan di {OUTPUT_REMOVED})")


✅ Total komentar bersih: 13045 (disimpan di cleaned_comments_filtered(fr1a).csv)
🗑️ Total komentar dibuang: 17145 (disimpan di log_removed_comments(fr1a).csv)
